In [ ]:
# =============================================================================
# CELL 1 — BUILD TRAIN-ONLY SYNTHETIC TEMPORAL GRAPH COMPANION
#
# PURPOSE:
# Prepare data for Node-aware LSTM and Graph + LSTM.
#
# IMPORTANT:
# Nodes and edges are SYNTHETIC, not recovered real IP connections.
# Targets represent NETWORK-WIDE future attack onset (H5, H10, H15).
#
# READS:  Frozen Synthetic_NAF_v2 TRAIN only
# WRITES: New GraphCompanion folder only
# TEST:   NOT ACCESSED
# =============================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

from google.colab import drive

# =============================================================================
# 1. GOOGLE DRIVE AND PROJECT PATHS
# =============================================================================

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

V2 = ROOT / "Synthetic_NAF_v2"

DATA = V2 / "release_candidate_corrected"

TRAIN = DATA / "windows_W10_TRAIN.npz"

CONTRACT_FILE = DATA / "FEATURE_AND_DATASET_CONTRACT.json"

HASHES = DATA / "SHA256SUMS.json"

# Separate output folder; original frozen V2 remains unchanged
OUT = (
    ROOT
    / "Synthetic_NAF_v2_GraphCompanion"
    / "TRAIN_only_v1"
)

GRAPH_FILE = OUT / "TRAIN_temporal_graphs.npz"

MANIFEST_FILE = OUT / "GRAPH_COMPANION_MANIFEST.json"

N_NODES = 8
GRAPH_SEED = 20261026

NODE_NAMES = [
    "flows_out",
    "forward_packets",
    "backward_packets",
    "forward_bytes",
    "backward_bytes",
    "packets_per_second",
    "bytes_per_second",
    "flow_duration_mean"
]


# =============================================================================
# 2. SHA256 INTEGRITY FUNCTION
# =============================================================================

def sha256(path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


# =============================================================================
# 3. VERIFY FROZEN DATASET
# =============================================================================

print("=" * 90)
print("GRAPH V2 — TRAIN-ONLY DATA BUILDER")
print("=" * 90)

for path in [TRAIN, CONTRACT_FILE, HASHES]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen TRAIN input missing: {path}"
        )

contract = json.loads(
    CONTRACT_FILE.read_text()
)

assert contract["history_minutes"] == 10
assert contract["forecast_horizons"] == [5, 10, 15]

features = list(contract["features"])

assert len(features) == 14

expected_sha = json.loads(
    HASHES.read_text()
)["windows_W10_TRAIN.npz"]

train_sha = sha256(TRAIN)

if train_sha != expected_sha:
    raise RuntimeError(
        "Frozen TRAIN SHA256 mismatch. STOP."
    )

print("Frozen TRAIN integrity: PASS")
print("Input features:", len(features))
print("History minutes:", contract["history_minutes"])
print("Forecast horizons:", contract["forecast_horizons"])


# =============================================================================
# 4. HANDLE EXISTING OUTPUT SAFELY
# =============================================================================

if GRAPH_FILE.exists() or MANIFEST_FILE.exists():

    if not (
        GRAPH_FILE.exists()
        and MANIFEST_FILE.exists()
    ):
        raise RuntimeError(
            "Partial output exists. Inspect before rerunning."
        )

    old = json.loads(
        MANIFEST_FILE.read_text()
    )

    if old.get("source_train_sha256") != train_sha:
        raise RuntimeError(
            "Existing graph companion uses a different TRAIN file."
        )

    if sha256(GRAPH_FILE) != old.get("artifact_sha256"):
        raise RuntimeError(
            "Existing graph companion checksum mismatch."
        )

    print("\nTRAIN graph companion already exists.")
    print("Checksum verification: PASS")
    print("No rebuild needed.")
    print("File:", GRAPH_FILE)
    print("Frames:", old["frames"])
    print("Windows:", old["windows"])

else:

    # =========================================================================
    # 5. LOAD FROZEN TRAIN ARRAYS
    # =========================================================================

    print("\nLoading frozen TRAIN arrays...")

    with np.load(TRAIN, allow_pickle=False) as pack:

        X = pack["X"].astype(np.float32)

        y = pack["y_horizons"].astype(np.int8)

        episodes = pack["episode_id"].astype(str)

        minutes = pack["minute"].astype(np.int32)

        onset = pack["onset_class"].astype(np.int16)

        family = pack["family_class"].astype(np.int8)

    assert X.ndim == 3
    assert X.shape[1:] == (10, 14)

    assert y.shape == (len(X), 3)

    assert len(episodes) == len(minutes) == len(X)

    assert np.isfinite(X).all()
    assert (X >= 0).all()

    print("TRAIN windows:", len(X))
    print("Sequence shape:", X.shape)
    print("Target shape:", y.shape)
    print("Unique episodes:", len(np.unique(episodes)))

    # =========================================================================
    # 6. FEATURE MAPPING
    # =========================================================================

    ix = {
        name: features.index(name)
        for name in [
            "Flow_Count",
            "Forward_Packets_Sum",
            "Backward_Packets_Sum",
            "Forward_Bytes_Sum",
            "Backward_Bytes_Sum",
            "Packets_Per_Second",
            "Bytes_Per_Second",
            "Flow_Duration_Mean"
        ]
    }

    # Group training windows by episode
    rows_by_episode = (
        pd.Series(np.arange(len(X)))
        .groupby(episodes, sort=True)
    )

    node_parts = []
    edge_parts = []

    end_frame = np.empty(
        len(X),
        dtype=np.int32
    )

    first_frame = 0
    n_eps = 0

    # =========================================================================
    # 7. CONSTRUCT SYNTHETIC TEMPORAL NETWORK GRAPHS
    # =========================================================================

    print("\nConstructing synthetic temporal graphs...")

    for episode_id, row_series in rows_by_episode:

        rows = row_series.to_numpy(dtype=np.int64)

        rows = rows[
            np.argsort(
                minutes[rows],
                kind="stable"
            )
        ]

        ts = minutes[rows]

        if (
            len(np.unique(ts)) != len(ts)
            or (
                len(ts) > 1
                and not np.all(np.diff(ts) == 1)
            )
        ):
            raise RuntimeError(
                f"Nonconsecutive or duplicate minutes: {episode_id}"
            )

        # ---------------------------------------------------------------------
        # Reconstruct historical observations from overlapping TRAIN windows
        # ---------------------------------------------------------------------

        stream = np.concatenate(
            (
                X[rows[0], :9],
                X[rows, -1]
            ),
            axis=0
        )

        for sample_idx in [
            0,
            len(rows) // 2,
            len(rows) - 1
        ]:

            if not np.allclose(
                X[rows[sample_idx]],
                stream[sample_idx:sample_idx + 10],
                rtol=1e-6,
                atol=1e-5
            ):
                raise RuntimeError(
                    f"Overlapping TRAIN windows disagree: {episode_id}"
                )

        # ---------------------------------------------------------------------
        # Episode-specific deterministic random generator
        # ---------------------------------------------------------------------

        digest = hashlib.sha256(
            f"{GRAPH_SEED}|{episode_id}".encode()
        ).digest()

        rng = np.random.default_rng(
            int.from_bytes(
                digest[:8],
                "little"
            )
        )

        # Synthetic allocation of network traffic across 8 nodes
        base_node_share = rng.dirichlet(
            np.full(N_NODES, 2.0)
        )

        # ---------------------------------------------------------------------
        # Create synthetic directed network connectivity
        # ---------------------------------------------------------------------

        permutation = rng.permutation(N_NODES)

        routing = np.zeros(
            (N_NODES, N_NODES),
            dtype=np.float32
        )

        for j, src in enumerate(permutation):

            dst1 = permutation[
                (j + 1) % N_NODES
            ]

            candidates = [
                k for k in range(N_NODES)
                if k not in (src, dst1)
            ]

            dst2 = int(
                rng.choice(candidates)
            )

            a = float(
                rng.uniform(0.35, 0.80)
            )

            routing[src, dst1] = a
            routing[src, dst2] = 1.0 - a

        assert np.allclose(
            routing.sum(axis=1),
            1.0
        )

        assert np.all(
            np.diag(routing) == 0
        )

        # ---------------------------------------------------------------------
        # Initialize per-minute graph arrays
        # ---------------------------------------------------------------------

        local_nodes = np.empty(
            (
                len(stream),
                N_NODES,
                len(NODE_NAMES)
            ),
            dtype=np.float32
        )

        local_edges = np.empty(
            (
                len(stream),
                N_NODES,
                N_NODES
            ),
            dtype=np.float32
        )

        # ---------------------------------------------------------------------
        # Generate node and edge features using observed traffic only
        # ---------------------------------------------------------------------

        for k, f in enumerate(stream):

            # Random variation independent of the future attack labels
            share = (
                0.90 * base_node_share
                + 0.10 * rng.dirichlet(
                    np.full(N_NODES, 2.0)
                )
            )

            share = (
                share / share.sum()
            ).astype(np.float32)

            values = np.asarray(
                [
                    f[ix["Flow_Count"]],
                    f[ix["Forward_Packets_Sum"]],
                    f[ix["Backward_Packets_Sum"]],
                    f[ix["Forward_Bytes_Sum"]],
                    f[ix["Backward_Bytes_Sum"]],
                    f[ix["Packets_Per_Second"]],
                    f[ix["Bytes_Per_Second"]]
                ],
                dtype=np.float32
            )

            # Traffic features assigned across synthetic nodes
            local_nodes[k, :, :7] = (
                share[:, None]
                * values[None, :]
            )

            # Mean flow duration retained as a global-style feature
            local_nodes[k, :, 7] = (
                f[ix["Flow_Duration_Mean"]]
            )

            # Weighted directed edges
            local_edges[k] = (
                f[ix["Flow_Count"]]
                * share[:, None]
                * routing
            )

            # Check that synthetic edge-flow totals are conserved
            if not np.isclose(
                local_edges[k].sum(),
                f[ix["Flow_Count"]],
                rtol=1e-5,
                atol=1e-3
            ):
                raise RuntimeError(
                    "Synthetic edge-flow conservation failed"
                )

        # ---------------------------------------------------------------------
        # Store episode graph sequences
        # ---------------------------------------------------------------------

        node_parts.append(local_nodes)
        edge_parts.append(local_edges)

        end_frame[rows] = (
            first_frame
            + 9
            + np.arange(
                len(rows),
                dtype=np.int32
            )
        )

        first_frame += len(stream)
        n_eps += 1

        if n_eps % 250 == 0:
            print(
                f"Built {n_eps} episodes / "
                f"{first_frame:,} unique minute snapshots",
                flush=True
            )

    # =========================================================================
    # 8. COMBINE GRAPH DATA
    # =========================================================================

    print("\nCombining graph snapshots...")

    node_frames = np.concatenate(
        node_parts,
        axis=0
    )

    edge_frames = np.concatenate(
        edge_parts,
        axis=0
    )

    assert (
        node_frames.shape[0]
        == edge_frames.shape[0]
        == first_frame
    )

    assert np.all(end_frame >= 9)

    assert np.isfinite(node_frames).all()
    assert np.isfinite(edge_frames).all()

    # =========================================================================
    # 9. VERIFY GRAPH FEATURES
    # =========================================================================

    print("\nVerifying generated graph data...")

    check_rng = np.random.default_rng(31)

    sampled_indices = check_rng.integers(
        0,
        len(X),
        size=min(256, len(X))
    )

    for i in sampled_indices:

        flow_sum = node_frames[
            end_frame[i],
            :,
            0
        ].sum()

        assert np.isclose(
            flow_sum,
            X[i, -1, ix["Flow_Count"]],
            rtol=1e-5,
            atol=1e-3
        )

    print("Graph feature consistency: PASS")

    # =========================================================================
    # 10. SAVE GRAPH COMPANION DATASET
    # =========================================================================

    OUT.mkdir(
        parents=True,
        exist_ok=True
    )

    tmp = OUT / "TRAIN_temporal_graphs.tmp.npz"

    # Store each graph snapshot only once.
    # Overlapping windows reference shared snapshots.

    np.savez_compressed(
        tmp,
        node_frames=node_frames,
        edge_flow_frames=edge_frames,
        window_end_frame=end_frame,
        y_horizons=y,
        onset_class=onset,
        family_class=family,
        episode_id=episodes,
        minute=minutes,
        node_feature_names=np.asarray(NODE_NAMES)
    )

    tmp.replace(GRAPH_FILE)

    # =========================================================================
    # 11. CREATE OUTPUT MANIFEST
    # =========================================================================

    manifest = {
        "version": "TRAIN_only_v1",
        "source": (
            "Frozen Synthetic_NAF_v2 "
            "training windows ONLY"
        ),
        "source_train_sha256": train_sha,
        "artifact_sha256": sha256(GRAPH_FILE),
        "windows": int(len(X)),
        "episodes": int(n_eps),
        "frames": int(first_frame),
        "nodes_per_graph": N_NODES,
        "node_feature_names": NODE_NAMES,
        "history_minutes": 10,
        "forecast_horizons_minutes": [5, 10, 15],
        "labels": (
            "network-wide future onset; "
            "no host-specific attack labels"
        ),
        "graph_origin": (
            "synthetically constructed from observed "
            "aggregate TRAIN features; NOT real IP edges"
        ),
        "seed": GRAPH_SEED,
        "window_lookup": (
            "For each end index e, use "
            "node_frames[e-9:e+1] and "
            "edge_flow_frames[e-9:e+1]"
        ),
        "validation_or_test_accessed": False
    }

    MANIFEST_FILE.write_text(
        json.dumps(
            manifest,
            indent=2
        )
    )

    # =========================================================================
    # 12. FINAL SUMMARY
    # =========================================================================

    print("\n" + "=" * 90)
    print("GRAPH COMPANION TRAIN BUILD: PASS")
    print("=" * 90)

    print("TRAIN windows:", len(X))
    print("Episodes:", n_eps)
    print("Unique minute graphs:", first_frame)

    print("Node tensor:", node_frames.shape)
    print("Edge tensor:", edge_frames.shape)

    print("\nSaved graph data:", GRAPH_FILE)
    print("Saved manifest:", MANIFEST_FILE)

    print("\nOriginal V2 files modified: NO")
    print("VALIDATION accessed: NO")
    print("TEST accessed: NO")
    print("Model training: NOT STARTED")

    print("=" * 90)

In [ ]:

# ============================================================
# CELL 2 — NODE-AWARE LSTM: TRAIN-ONLY MODEL
#
# Input: 10 minutes x 8 simulated nodes x 8 features
# Output: H5, H10, H15 future-attack probabilities
#
# Does not read VALIDATION or TEST.
# Does not overwrite existing completed models.
# Saves training progress after every epoch.
# ============================================================

from pathlib import Path
import json
import hashlib
import os
import random
import numpy as np
import torch
import torch.nn as nn

from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

GRAPH_ROOT = ROOT / "Synthetic_NAF_v2_GraphCompanion"

DATA_DIR = GRAPH_ROOT / "TRAIN_only_v1"
GRAPH_FILE = DATA_DIR / "TRAIN_temporal_graphs.npz"
MANIFEST_FILE = DATA_DIR / "GRAPH_COMPANION_MANIFEST.json"

MODEL_DIR = (
    GRAPH_ROOT / "models" /
    "NodeAware_LSTM" / "train_only_v1"
)

FINAL_FILE = MODEL_DIR / "NODE_LSTM_STATE.pt"
PROGRESS_FILE = MODEL_DIR / "NODE_LSTM_PROGRESS.pt"
RECORD_FILE = MODEL_DIR / "NODE_LSTM_TRAIN_RECORD.json"

SEED = 20261027
EPOCHS = 12
BATCH_SIZE = 256
LEARNING_RATE = 0.001
HIDDEN_DIM = 32

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
else:
    torch.set_num_threads(
        min(4, os.cpu_count() or 1)
    )

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

print("=" * 80)
print("NODE-AWARE LSTM — TRAINING")
print("=" * 80)
print("Device:", DEVICE)

# ============================================================
# 1. VERIFY GRAPH DATA
# ============================================================

for path in [GRAPH_FILE, MANIFEST_FILE]:
    if not path.is_file():
        raise FileNotFoundError(path)

manifest = json.loads(MANIFEST_FILE.read_text())
graph_sha = sha256(GRAPH_FILE)

assert graph_sha == manifest["artifact_sha256"]
assert manifest["history_minutes"] == 10
assert manifest["nodes_per_graph"] == 8
assert manifest["forecast_horizons_minutes"] == [5, 10, 15]

print("Graph companion SHA256: PASS")

with np.load(GRAPH_FILE, allow_pickle=False) as data:
    node_frames = data["node_frames"].astype(np.float32)
    end_frames = data["window_end_frame"].astype(np.int64)
    targets = data["y_horizons"].astype(np.float32)
    episodes = data["episode_id"].astype(str)

assert node_frames.ndim == 3
assert node_frames.shape[1:] == (8, 8)
assert targets.shape == (len(end_frames), 3)
assert len(episodes) == len(end_frames)
assert end_frames.min() >= 9
assert end_frames.max() < len(node_frames)
assert np.isfinite(node_frames).all()
assert np.all(node_frames >= 0)
assert np.isin(targets, [0, 1]).all()
assert np.all(targets[:, 0] <= targets[:, 1])
assert np.all(targets[:, 1] <= targets[:, 2])

print("Windows:", len(end_frames))
print("Episodes:", len(np.unique(episodes)))
print("Node frames:", node_frames.shape)
print("Targets:", targets.shape)

# ============================================================
# 2. TRAIN-ONLY FEATURE SCALING
# ============================================================

# Fit preprocessing exclusively on TRAIN node frames.
log_frames = np.log1p(node_frames)
flat = log_frames.reshape(-1, 8)

median = np.median(flat, axis=0).astype(np.float32)
q25 = np.percentile(flat, 25, axis=0)
q75 = np.percentile(flat, 75, axis=0)

scale = np.maximum(
    q75 - q25, 0.05
).astype(np.float32)

scaled_frames = np.clip(
    (log_frames - median) / scale,
    -8.0,
    8.0
).astype(np.float32)

del log_frames, flat, node_frames

print("Preprocessing: LOG1P + MEDIAN/IQR + CLIP8")
print("Scaler fitted on TRAIN only: YES")

# ============================================================
# 3. NODE-AWARE LSTM ARCHITECTURE
# ============================================================

class NodeAwareLSTM(nn.Module):

    def __init__(self, input_dim=8, hidden_dim=32):
        super().__init__()

        # The same LSTM processes each synthetic node.
        self.node_lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        # Learn which node representations contribute
        # to the network-wide forecast.
        self.attention = nn.Linear(hidden_dim, 1)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 3)
        )

    def forward(self, x):
        # x shape: (batch, time=10, nodes=8, features=8)
        b, t, n, f = x.shape

        x = x.permute(0, 2, 1, 3)
        x = x.reshape(b * n, t, f)

        _, (hidden, _) = self.node_lstm(x)

        node_embeddings = hidden[-1].reshape(
            b, n, -1
        )

        weights = torch.softmax(
            self.attention(node_embeddings),
            dim=1
        )

        network_embedding = (
            node_embeddings * weights
        ).sum(dim=1)

        logits = self.head(network_embedding)

        # Conditional hazards for 0–5, 5–10, 10–15 min.
        hazards = torch.sigmoid(logits)

        # Cumulative probabilities are guaranteed monotonic.
        probabilities = 1.0 - torch.cumprod(
            1.0 - hazards,
            dim=1
        )

        return probabilities.clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )

model = NodeAwareLSTM(
    input_dim=8,
    hidden_dim=HIDDEN_DIM
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.001
)

criterion = nn.BCELoss()

print("\nArchitecture: NODE-AWARE LSTM")
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters())
)

# ============================================================
# 4. QUICK INFERENCE SHAPE CHECK
# ============================================================

model.eval()

with torch.no_grad():
    dummy = torch.zeros(
        2, 10, 8, 8,
        device=DEVICE
    )

    prediction = model(dummy)

    assert prediction.shape == (2, 3)
    assert torch.all(
        prediction[:, 0] <= prediction[:, 1]
    )
    assert torch.all(
        prediction[:, 1] <= prediction[:, 2]
    )

print("Model output/monotonicity check: PASS")

# ============================================================
# 5. CHECKPOINT RECOVERY
# ============================================================

MODEL_DIR.mkdir(parents=True, exist_ok=True)

start_epoch = 0
loss_history = []

if FINAL_FILE.exists():
    saved = torch.load(
        FINAL_FILE,
        map_location="cpu",
        weights_only=False
    )

    if saved.get("graph_sha256") != graph_sha:
        raise RuntimeError(
            "Existing final model belongs to another graph dataset."
        )

    print("\nCompleted model already exists:")
    print(FINAL_FILE)
    print("Checkpoint SHA256:", sha256(FINAL_FILE))
    print("No retraining or overwrite performed.")

else:

    if PROGRESS_FILE.exists():

        saved = torch.load(
            PROGRESS_FILE,
            map_location=DEVICE,
            weights_only=False
        )

        if saved.get("graph_sha256") != graph_sha:
            raise RuntimeError(
                "Progress checkpoint dataset mismatch."
            )

        if not np.array_equal(saved["median"], median):
            raise RuntimeError("Scaler median mismatch.")

        if not np.array_equal(saved["scale"], scale):
            raise RuntimeError("Scaler IQR mismatch.")

        model.load_state_dict(saved["model_state"])
        optimizer.load_state_dict(saved["optimizer_state"])

        start_epoch = int(saved["epoch"])
        loss_history = list(saved["loss_history"])

        print(
            "\nResuming from completed epoch:",
            start_epoch
        )

    # ========================================================
    # 6. TRAIN
    # ========================================================

    n = len(end_frames)
    offsets = np.arange(-9, 1, dtype=np.int64)

    print("\nStarting training...")
    print("Total epochs:", EPOCHS)
    print("Batch size:", BATCH_SIZE)

    for epoch in range(start_epoch, EPOCHS):

        model.train()

        rng = np.random.default_rng(SEED + epoch)
        shuffled = rng.permutation(n)

        total_loss = 0.0
        seen = 0

        for start in range(0, n, BATCH_SIZE):

            ids = shuffled[start:start + BATCH_SIZE]

            frame_indices = (
                end_frames[ids, None]
                + offsets[None, :]
            )

            xb = torch.from_numpy(
                scaled_frames[frame_indices]
            ).to(DEVICE)

            yb = torch.from_numpy(
                targets[ids]
            ).to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            probs = model(xb)

            loss = criterion(probs, yb)

            loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_loss += loss.item() * len(ids)
            seen += len(ids)

        epoch_loss = total_loss / seen
        loss_history.append(epoch_loss)

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} "
            f"| TRAIN BCE={epoch_loss:.6f}",
            flush=True
        )

        # Atomic progress checkpoint after every epoch.
        progress = {
            "epoch": epoch + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "loss_history": loss_history,
            "median": median,
            "scale": scale,
            "graph_sha256": graph_sha
        }

        temp_path = MODEL_DIR / "NODE_LSTM_PROGRESS.tmp"

        torch.save(progress, temp_path)
        temp_path.replace(PROGRESS_FILE)

    # ========================================================
    # 7. SAVE FINAL TRAINED MODEL
    # ========================================================

    final_record = {
        "model": "NodeAwareLSTM",
        "input_shape": [10, 8, 8],
        "hidden_dim": HIDDEN_DIM,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "graph_sha256": graph_sha,
        "training_loss": loss_history,
        "preprocessing": "LOG1P_MEDIAN_IQR_CLIP8",
        "data_split": "TRAIN_ONLY",
        "graph_origin": "synthetic_nodes_and_edges",
        "test_accessed": False
    }

    final_state = {
        "model_state": model.state_dict(),
        "median": median,
        "scale": scale,
        "record": final_record,
        "graph_sha256": graph_sha
    }

    temp_final = MODEL_DIR / "NODE_LSTM_FINAL.tmp"

    torch.save(final_state, temp_final)
    temp_final.replace(FINAL_FILE)

    RECORD_FILE.write_text(
        json.dumps(final_record, indent=2)
    )

    print("\n" + "=" * 80)
    print("NODE-AWARE LSTM — TRAINING COMPLETE")
    print("=" * 80)
    print("Checkpoint:", FINAL_FILE)
    print("Checkpoint SHA256:", sha256(FINAL_FILE))
    print("Training windows:", n)
    print("Final TRAIN BCE:", loss_history[-1])
    print("Official VALIDATION accessed: NO")
    print("Official TEST accessed: NO")
    print("=" * 80)


In [ ]:

# ================================================================
# CELL 3 — GRAPH NEURAL NETWORK + LSTM
# TRAIN-ONLY FORECASTING MODEL
#
# Architecture:
# Weighted Directed Graph Convolution -> Graph Convolution
# -> Graph Pooling -> LSTM -> H5/H10/H15 Forecast
#
# No additional graph package installation required.
# Uses original frozen V2 TRAIN labels only.
# Synthetic node identities and synthetic edge relationships.
#
# VALIDATION: NOT ACCESSED
# TEST: NOT ACCESSED
# ================================================================

from pathlib import Path
import hashlib
import json
import random
import os
import time

import numpy as np
import torch
import torch.nn as nn

from google.colab import drive

# ================================================================
# 1. CONFIGURATION
# ================================================================

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

GRAPH_ROOT = ROOT / "Synthetic_NAF_v2_GraphCompanion"

DATA_DIR = GRAPH_ROOT / "TRAIN_only_v1"

GRAPH_FILE = DATA_DIR / "TRAIN_temporal_graphs.npz"
MANIFEST_FILE = DATA_DIR / "GRAPH_COMPANION_MANIFEST.json"

MODEL_DIR = (
    GRAPH_ROOT / "models" /
    "Graph_LSTM" / "train_only_v1"
)

FINAL_FILE = MODEL_DIR / "GRAPH_LSTM_STATE.pt"
PROGRESS_FILE = MODEL_DIR / "GRAPH_LSTM_PROGRESS.pt"
RECORD_FILE = MODEL_DIR / "GRAPH_LSTM_TRAIN_RECORD.json"

SEED = 20261027

EPOCHS = 12
BATCH_SIZE = 256
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.001

GRAPH_HIDDEN = 32
LSTM_HIDDEN = 64
DROPOUT = 0.1

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
else:
    torch.set_num_threads(
        min(4, os.cpu_count() or 1)
    )

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

CONFIG = {
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "graph_hidden": GRAPH_HIDDEN,
    "lstm_hidden": LSTM_HIDDEN,
    "dropout": DROPOUT,
    "input_features": 8,
    "nodes": 8,
    "history": 10
}

print("=" * 85)
print("GRAPH + LSTM — TRAINING")
print("=" * 85)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

# ================================================================
# 2. VERIFY FROZEN GRAPH COMPANION
# ================================================================

for p in [GRAPH_FILE, MANIFEST_FILE]:
    if not p.is_file():
        raise FileNotFoundError(p)

manifest = json.loads(
    MANIFEST_FILE.read_text(encoding="utf-8")
)

graph_sha = sha256(GRAPH_FILE)

assert graph_sha == manifest["artifact_sha256"]
assert manifest["nodes_per_graph"] == 8
assert manifest["history_minutes"] == 10
assert manifest["forecast_horizons_minutes"] == [5, 10, 15]

print("\nGraph companion integrity: PASS")

with np.load(GRAPH_FILE, allow_pickle=False) as data:

    node_frames = data["node_frames"].astype(np.float32)

    edge_frames = data["edge_flow_frames"].astype(np.float32)

    end_frames = data["window_end_frame"].astype(np.int64)

    targets = data["y_horizons"].astype(np.float32)

    episodes = data["episode_id"].astype(str)

assert node_frames.shape[1:] == (8, 8)
assert edge_frames.shape == (
    len(node_frames), 8, 8
)
assert targets.shape == (len(end_frames), 3)
assert len(end_frames) == len(episodes)

assert end_frames.min() >= 9
assert end_frames.max() < len(node_frames)

assert np.isfinite(node_frames).all()
assert np.isfinite(edge_frames).all()

assert (node_frames >= 0).all()
assert (edge_frames >= 0).all()

assert np.isin(targets, [0, 1]).all()

assert np.all(targets[:, 0] <= targets[:, 1])
assert np.all(targets[:, 1] <= targets[:, 2])

print("Training windows:", len(end_frames))
print("Unique episodes:", len(np.unique(episodes)))
print("Node frame shape:", node_frames.shape)
print("Edge frame shape:", edge_frames.shape)
print("Targets:", targets.shape)

# ================================================================
# 3. TRAIN-ONLY FEATURE PREPROCESSING
# ================================================================

print("\nFitting TRAIN-only preprocessing...")

# Node traffic features
log_nodes = np.log1p(node_frames)

flat = log_nodes.reshape(-1, 8)

median = np.median(
    flat, axis=0
).astype(np.float32)

q25 = np.percentile(flat, 25, axis=0)
q75 = np.percentile(flat, 75, axis=0)

scale = np.maximum(
    q75 - q25, 0.05
).astype(np.float32)

scaled_nodes = np.clip(
    (log_nodes - median) / scale,
    -8.0,
    8.0
).astype(np.float32)

del log_nodes, flat, node_frames

# Normalize edge flows per graph.
# The graph contains weighted directed connections.
# This keeps edge values numerically stable while
# preserving relative communication strength.

edge_total = edge_frames.sum(
    axis=(1, 2),
    keepdims=True
)

scaled_edges = (
    edge_frames /
    np.maximum(edge_total, 1e-8)
).astype(np.float32)

del edge_frames, edge_total

assert np.isfinite(scaled_nodes).all()
assert np.isfinite(scaled_edges).all()

print("Node preprocessing: LOG1P / MEDIAN-IQR / CLIP8")
print("Edge preprocessing: PER-GRAPH FLOW FRACTIONS")
print("Scaler fitted using TRAIN only: YES")

# ================================================================
# 4. WEIGHTED DIRECTED GRAPH CONVOLUTION
# ================================================================

class WeightedGraphConv(nn.Module):

    def __init__(self, in_dim, out_dim):
        super().__init__()

        self.self_linear = nn.Linear(
            in_dim, out_dim
        )

        self.neighbor_linear = nn.Linear(
            in_dim, out_dim
        )

    def forward(self, x, adjacency):

        # x:         (batch_graphs, nodes, features)
        # adjacency: (batch_graphs, target, source)
        #
        # Incoming neighbors are aggregated for each node.

        incoming = torch.bmm(
            adjacency, x
        )

        result = (
            self.self_linear(x)
            + self.neighbor_linear(incoming)
        )

        return torch.relu(result)


# ================================================================
# 5. GRAPH + LSTM ARCHITECTURE
# ================================================================

class GraphLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_dim=8,
        graph_hidden=32,
        lstm_hidden=64,
        dropout=0.1
    ):
        super().__init__()

        # First graph convolution
        self.graph_conv1 = WeightedGraphConv(
            input_dim,
            graph_hidden
        )

        # Second graph convolution
        self.graph_conv2 = WeightedGraphConv(
            graph_hidden,
            graph_hidden
        )

        self.graph_dropout = nn.Dropout(
            dropout
        )

        # Two pooled graph vectors:
        # mean node embedding + max node embedding

        self.lstm = nn.LSTM(
            input_size=graph_hidden * 2,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    def forward(self, nodes, edges):

        # nodes:
        #   (batch, time=10, nodes=8, features=8)
        #
        # edges:
        #   (batch, time=10, source=8, target=8)

        b, t, n, f = nodes.shape

        x = nodes.reshape(
            b * t, n, f
        )

        adjacency = edges.reshape(
            b * t, n, n
        )

        # Input convention: [source, target].
        # Graph convolution needs [target, source].

        adjacency = adjacency.transpose(1, 2)

        # Incoming edge normalization.
        # A node with no incoming edges gets zero
        # incoming messages, while keeping its own
        # representation via self_linear.

        degree = adjacency.sum(
            dim=-1,
            keepdim=True
        )

        adjacency = adjacency / degree.clamp(
            min=1e-8
        )

        # Spatial graph encoder

        h = self.graph_conv1(
            x, adjacency
        )

        h = self.graph_dropout(h)

        h = self.graph_conv2(
            h, adjacency
        )

        # Aggregate node embeddings into one
        # graph representation per minute.

        mean_pool = h.mean(dim=1)
        max_pool = h.max(dim=1).values

        graph_embedding = torch.cat(
            [mean_pool, max_pool],
            dim=-1
        )

        # Recover temporal dimension

        graph_sequence = graph_embedding.reshape(
            b, t, -1
        )

        # Temporal LSTM

        temporal_output, _ = self.lstm(
            graph_sequence
        )

        last_state = temporal_output[:, -1, :]

        logits = self.head(last_state)

        # Conditional hazards for time intervals
        # 0–5, 5–10, 10–15 minutes.

        hazards = torch.sigmoid(logits)

        # Convert interval hazards to cumulative
        # probabilities, enforcing H5 <= H10 <= H15.

        probabilities = 1.0 - torch.cumprod(
            1.0 - hazards,
            dim=1
        )

        return probabilities.clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )


# ================================================================
# 6. INITIALIZE GRAPH + LSTM
# ================================================================

model = GraphLSTMForecaster(
    input_dim=8,
    graph_hidden=GRAPH_HIDDEN,
    lstm_hidden=LSTM_HIDDEN,
    dropout=DROPOUT
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

criterion = nn.BCELoss()

parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nMODEL ARCHITECTURE")
print(model)
print("\nTrainable parameters:", parameter_count)

# ================================================================
# 7. FORWARD-PASS AND MONOTONICITY TEST
# ================================================================

model.eval()

with torch.no_grad():

    dummy_nodes = torch.rand(
        2, 10, 8, 8,
        device=DEVICE
    )

    dummy_edges = torch.rand(
        2, 10, 8, 8,
        device=DEVICE
    )

    output = model(
        dummy_nodes,
        dummy_edges
    )

    assert output.shape == (2, 3)

    assert torch.all(
        output[:, 0] <= output[:, 1]
    )

    assert torch.all(
        output[:, 1] <= output[:, 2]
    )

    assert torch.isfinite(output).all()

print("\nGraph + LSTM forward pass: PASS")
print("Horizon monotonicity: PASS")

# ================================================================
# 8. CHECKPOINT RECOVERY
# ================================================================

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

start_epoch = 0
loss_history = []

if FINAL_FILE.exists():

    saved = torch.load(
        FINAL_FILE,
        map_location="cpu",
        weights_only=False
    )

    if saved.get("graph_sha256") != graph_sha:
        raise RuntimeError(
            "Existing final model uses a different graph dataset."
        )

    if saved.get("config") != CONFIG:
        raise RuntimeError(
            "Existing final model configuration differs."
        )

    print("\nGRAPH + LSTM ALREADY TRAINED")
    print("Checkpoint:", FINAL_FILE)
    print("SHA256:", sha256(FINAL_FILE))
    print("No model overwritten.")

else:

    if PROGRESS_FILE.exists():

        progress = torch.load(
            PROGRESS_FILE,
            map_location=DEVICE,
            weights_only=False
        )

        if progress["graph_sha256"] != graph_sha:
            raise RuntimeError(
                "Progress checkpoint dataset mismatch."
            )

        if progress["config"] != CONFIG:
            raise RuntimeError(
                "Progress checkpoint config mismatch."
            )

        if not np.array_equal(
            progress["median"], median
        ):
            raise RuntimeError(
                "Preprocessing median mismatch."
            )

        if not np.array_equal(
            progress["scale"], scale
        ):
            raise RuntimeError(
                "Preprocessing scale mismatch."
            )

        model.load_state_dict(
            progress["model_state"]
        )

        optimizer.load_state_dict(
            progress["optimizer_state"]
        )

        start_epoch = int(
            progress["epoch"]
        )

        loss_history = list(
            progress["loss_history"]
        )

        print(
            "\nResuming from completed epoch:",
            start_epoch
        )

    # ============================================================
    # 9. TRAINING
    # ============================================================

    n_windows = len(end_frames)

    offsets = np.arange(
        -9, 1,
        dtype=np.int64
    )

    print("\n" + "=" * 85)
    print("STARTING GRAPH + LSTM TRAINING")
    print("=" * 85)

    print("Windows:", n_windows)
    print("Epochs:", EPOCHS)
    print("Batch size:", BATCH_SIZE)
    print("Device:", DEVICE)

    for epoch in range(
        start_epoch, EPOCHS
    ):

        start_time = time.time()

        model.train()

        rng = np.random.default_rng(
            SEED + epoch
        )

        order = rng.permutation(
            n_windows
        )

        total_loss = 0.0
        total_seen = 0

        for start in range(
            0, n_windows, BATCH_SIZE
        ):

            ids = order[
                start:start + BATCH_SIZE
            ]

            indices = (
                end_frames[ids, None]
                + offsets[None, :]
            )

            xb_nodes = torch.from_numpy(
                scaled_nodes[indices]
            ).to(DEVICE)

            xb_edges = torch.from_numpy(
                scaled_edges[indices]
            ).to(DEVICE)

            yb = torch.from_numpy(
                targets[ids]
            ).to(DEVICE)

            optimizer.zero_grad(
                set_to_none=True
            )

            predictions = model(
                xb_nodes,
                xb_edges
            )

            loss = criterion(
                predictions,
                yb
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss at epoch {epoch + 1}"
                )

            loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_loss += (
                loss.item() * len(ids)
            )

            total_seen += len(ids)

        epoch_loss = (
            total_loss / total_seen
        )

        loss_history.append(
            epoch_loss
        )

        duration = time.time() - start_time

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS}"
            f" | TRAIN BCE={epoch_loss:.6f}"
            f" | seconds={duration:.1f}",
            flush=True
        )

        # ========================================================
        # 10. SAVE PROGRESS AFTER EACH EPOCH
        # ========================================================

        progress = {
            "epoch": epoch + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "loss_history": loss_history,
            "median": median,
            "scale": scale,
            "graph_sha256": graph_sha,
            "config": CONFIG,
            "torch_rng_state": torch.get_rng_state()
        }

        if torch.cuda.is_available():
            progress["cuda_rng_state"] = (
                torch.cuda.get_rng_state_all()
            )

        temp_progress = (
            MODEL_DIR / "GRAPH_LSTM_PROGRESS.tmp"
        )

        torch.save(
            progress,
            temp_progress
        )

        temp_progress.replace(
            PROGRESS_FILE
        )

    # ============================================================
    # 11. SAVE FINAL MODEL
    # ============================================================

    final_state = {
        "model_state": model.state_dict(),
        "median": median,
        "scale": scale,
        "graph_sha256": graph_sha,
        "config": CONFIG,
        "loss_history": loss_history,
        "architecture": "WeightedGraphConv_GraphConv_LSTM",
        "graph_origin": "SYNTHETIC",
        "data_split": "TRAIN_ONLY",
        "official_test_accessed": False
    }

    temp_final = (
        MODEL_DIR / "GRAPH_LSTM_FINAL.tmp"
    )

    torch.save(
        final_state,
        temp_final
    )

    temp_final.replace(
        FINAL_FILE
    )

    record = {
        "model": "GraphLSTMForecaster",
        "architecture": "WeightedGraphConv_GraphConv_LSTM",
        "config": CONFIG,
        "parameters": parameter_count,
        "train_windows": n_windows,
        "train_episodes": int(
            len(np.unique(episodes))
        ),
        "graph_sha256": graph_sha,
        "checkpoint_sha256": sha256(FINAL_FILE),
        "train_bce_history": loss_history,
        "final_train_bce": loss_history[-1],
        "preprocessing": "TRAIN_LOG1P_MEDIAN_IQR_CLIP8",
        "edge_processing": "ROW_NORMALIZED_INCOMING_FLOW",
        "source": "Synthetic_NAF_v2_GraphCompanion",
        "graph_origin": "Synthetic node and edge allocations",
        "validation_accessed": False,
        "test_accessed": False
    }

    RECORD_FILE.write_text(
        json.dumps(record, indent=2)
    )

    print("\n" + "=" * 85)
    print("GRAPH + LSTM — TRAINING COMPLETE")
    print("=" * 85)

    print("Model:", FINAL_FILE)
    print("Checkpoint SHA256:", sha256(FINAL_FILE))

    print("Trainable parameters:", parameter_count)
    print("Training windows:", n_windows)
    print("Training episodes:", len(np.unique(episodes)))
    print("Final TRAIN BCE:", loss_history[-1])

    print("\nOriginal V2 dataset modified: NO")
    print("Official VALIDATION accessed: NO")
    print("Official TEST accessed: NO")

    print("=" * 85)




In [ ]:

# ================================================================
# CELL 4 — BUILD SYNTHETIC GRAPH VALIDATION COMPANION
#
# Same deterministic generator as TRAIN.
# Uses frozen V2 VALIDATION, never TEST.
# No training, no model changes.
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/network-attack-forecasting")
V2 = ROOT / "Synthetic_NAF_v2"
DATA = V2 / "release_candidate_corrected"
GRAPH_ROOT = ROOT / "Synthetic_NAF_v2_GraphCompanion"

SOURCE = DATA / "windows_W10_VALIDATION.npz"
CONTRACT_FILE = DATA / "FEATURE_AND_DATASET_CONTRACT.json"
HASH_FILE = DATA / "SHA256SUMS.json"

TRAIN_MANIFEST_FILE = (
    GRAPH_ROOT / "TRAIN_only_v1" /
    "GRAPH_COMPANION_MANIFEST.json"
)

OUT = GRAPH_ROOT / "VALIDATION_only_v1"
GRAPH_FILE = OUT / "VALIDATION_temporal_graphs.npz"
MANIFEST_FILE = OUT / "GRAPH_COMPANION_VALIDATION_MANIFEST.json"

N_NODES = 8

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("=" * 85)
print("GRAPH COMPANION — OFFICIAL VALIDATION BUILDER")
print("=" * 85)

for p in [SOURCE, CONTRACT_FILE, HASH_FILE, TRAIN_MANIFEST_FILE]:
    if not p.is_file():
        raise FileNotFoundError(p)

contract = json.loads(CONTRACT_FILE.read_text())
hashes = json.loads(HASH_FILE.read_text())
train_manifest = json.loads(TRAIN_MANIFEST_FILE.read_text())

assert contract["history_minutes"] == 10
assert contract["forecast_horizons"] == [5, 10, 15]
assert train_manifest["nodes_per_graph"] == N_NODES
assert train_manifest["history_minutes"] == 10
assert train_manifest["forecast_horizons_minutes"] == [5, 10, 15]

SEED = int(train_manifest["seed"])
features = list(contract["features"])

assert len(features) == 14

source_sha = file_sha256(SOURCE)

if source_sha != hashes["windows_W10_VALIDATION.npz"]:
    raise RuntimeError("Frozen VALIDATION checksum mismatch.")

print("Frozen VALIDATION SHA256: PASS")
print("Using TRAIN graph generator seed:", SEED)

# ================================================================
# SAFE EXISTING OUTPUT CHECK
# ================================================================

if GRAPH_FILE.exists() or MANIFEST_FILE.exists():

    if not (GRAPH_FILE.exists() and MANIFEST_FILE.exists()):
        raise RuntimeError(
            "Incomplete existing VALIDATION output. "
            "Inspect before rebuilding."
        )

    saved_manifest = json.loads(MANIFEST_FILE.read_text())

    assert saved_manifest["source_validation_sha256"] == source_sha
    assert saved_manifest["generator_seed"] == SEED
    assert saved_manifest["artifact_sha256"] == file_sha256(GRAPH_FILE)

    print("\nVALIDATION companion already exists.")
    print("Existing integrity: PASS")
    print("No rebuild or overwrite performed.")

else:

    # ============================================================
    # LOAD FROZEN VALIDATION DATA
    # ============================================================

    with np.load(SOURCE, allow_pickle=False) as pack:
        X = pack["X"].astype(np.float32)
        y = pack["y_horizons"].astype(np.int8)
        episodes = pack["episode_id"].astype(str)
        minutes = pack["minute"].astype(np.int32)
        onset = pack["onset_class"].astype(np.int16)
        family = pack["family_class"].astype(np.int8)

    assert X.ndim == 3
    assert X.shape[1:] == (10, 14)
    assert y.shape == (len(X), 3)
    assert len(episodes) == len(minutes) == len(X)
    assert np.isfinite(X).all()
    assert (X >= 0).all()
    assert np.isin(y, [0, 1]).all()
    assert np.all(y[:, 0] <= y[:, 1])
    assert np.all(y[:, 1] <= y[:, 2])

    print("\nValidation windows:", len(X))
    print("Validation episodes:", len(np.unique(episodes)))
    print("Input shape:", X.shape)

    # ============================================================
    # IDENTICAL FEATURE MAPPING TO TRAIN BUILDER
    # ============================================================

    used_features = [
        "Flow_Count",
        "Forward_Packets_Sum",
        "Backward_Packets_Sum",
        "Forward_Bytes_Sum",
        "Backward_Bytes_Sum",
        "Packets_Per_Second",
        "Bytes_Per_Second",
        "Flow_Duration_Mean"
    ]

    ix = {name: features.index(name) for name in used_features}

    node_parts = []
    edge_parts = []
    end_frames = np.empty(len(X), dtype=np.int32)

    cursor = 0
    episode_count = 0

    print("\nGenerating deterministic VALIDATION graphs...")

    grouped_episodes = (
        pd.Series(np.arange(len(X)))
        .groupby(episodes, sort=True)
    )

    for episode_id, row_series in grouped_episodes:

        rows = row_series.to_numpy(dtype=np.int64)

        rows = rows[
            np.argsort(minutes[rows], kind="stable")
        ]

        ts = minutes[rows]

        if len(np.unique(ts)) != len(ts):
            raise RuntimeError(f"Duplicate minutes: {episode_id}")

        if len(ts) > 1 and not np.all(np.diff(ts) == 1):
            raise RuntimeError(f"Missing minutes: {episode_id}")

        # Reconstruct observed minute stream
        stream = np.concatenate(
            [X[rows[0], :9], X[rows, -1]],
            axis=0
        )

        # Verify reconstruction against saved input windows
        for k in [0, len(rows) // 2, len(rows) - 1]:
            if not np.allclose(
                X[rows[k]],
                stream[k:k + 10],
                rtol=1e-6,
                atol=1e-5
            ):
                raise RuntimeError(
                    f"Window reconstruction mismatch: {episode_id}"
                )

        # Same per-episode RNG derivation as TRAIN
        digest = hashlib.sha256(
            f"{SEED}|{episode_id}".encode()
        ).digest()

        rng = np.random.default_rng(
            int.from_bytes(digest[:8], "little")
        )

        base_node_share = rng.dirichlet(
            np.full(N_NODES, 2.0)
        )

        permutation = rng.permutation(N_NODES)

        routing = np.zeros(
            (N_NODES, N_NODES),
            dtype=np.float32
        )

        for j, src in enumerate(permutation):

            dst1 = permutation[(j + 1) % N_NODES]

            candidates = [
                k for k in range(N_NODES)
                if k not in (src, dst1)
            ]

            dst2 = int(rng.choice(candidates))
            a = float(rng.uniform(0.35, 0.80))

            routing[src, dst1] = a
            routing[src, dst2] = 1.0 - a

        assert np.allclose(routing.sum(axis=1), 1.0)
        assert np.all(np.diag(routing) == 0)

        local_nodes = np.empty(
            (len(stream), 8, 8),
            dtype=np.float32
        )

        local_edges = np.empty(
            (len(stream), 8, 8),
            dtype=np.float32
        )

        # ========================================================
        # CONSTRUCT OBSERVED NODE AND EDGE FEATURES
        # ========================================================

        for k, f in enumerate(stream):

            share = (
                0.90 * base_node_share
                + 0.10 * rng.dirichlet(
                    np.full(N_NODES, 2.0)
                )
            )

            share = (
                share / share.sum()
            ).astype(np.float32)

            values = np.asarray(
                [
                    f[ix["Flow_Count"]],
                    f[ix["Forward_Packets_Sum"]],
                    f[ix["Backward_Packets_Sum"]],
                    f[ix["Forward_Bytes_Sum"]],
                    f[ix["Backward_Bytes_Sum"]],
                    f[ix["Packets_Per_Second"]],
                    f[ix["Bytes_Per_Second"]]
                ],
                dtype=np.float32
            )

            local_nodes[k, :, :7] = (
                share[:, None] * values[None, :]
            )

            local_nodes[k, :, 7] = (
                f[ix["Flow_Duration_Mean"]]
            )

            local_edges[k] = (
                f[ix["Flow_Count"]]
                * share[:, None]
                * routing
            )

            if not np.isclose(
                local_edges[k].sum(),
                f[ix["Flow_Count"]],
                rtol=1e-5,
                atol=1e-3
            ):
                raise RuntimeError(
                    "Synthetic edge conservation failed."
                )

        node_parts.append(local_nodes)
        edge_parts.append(local_edges)

        end_frames[rows] = (
            cursor + 9 + np.arange(len(rows), dtype=np.int32)
        )

        cursor += len(stream)
        episode_count += 1

        if episode_count % 50 == 0:
            print(
                f"Built {episode_count} episodes / "
                f"{cursor:,} graph frames",
                flush=True
            )

    # ============================================================
    # COMBINE AND VERIFY
    # ============================================================

    node_frames = np.concatenate(node_parts, axis=0)
    edge_frames = np.concatenate(edge_parts, axis=0)

    assert node_frames.shape == (cursor, 8, 8)
    assert edge_frames.shape == (cursor, 8, 8)
    assert len(end_frames) == len(X)

    assert np.isfinite(node_frames).all()
    assert np.isfinite(edge_frames).all()
    assert end_frames.min() >= 9
    assert end_frames.max() < cursor

    # Sample consistency against original frozen inputs
    check_rng = np.random.default_rng(31)

    sample_ids = check_rng.integers(
        0, len(X), size=min(256, len(X))
    )

    for i in sample_ids:
        observed_flow = node_frames[
            end_frames[i], :, 0
        ].sum()

        expected_flow = X[
            i, -1, ix["Flow_Count"]
        ]

        assert np.isclose(
            observed_flow,
            expected_flow,
            rtol=1e-5,
            atol=1e-3
        )

    print("\nGraph feature integrity: PASS")

    # ============================================================
    # SAVE NEW VALIDATION-ONLY ARTIFACT
    # ============================================================

    OUT.mkdir(parents=True, exist_ok=True)

    temp_file = OUT / "VALIDATION_temporal_graphs.tmp.npz"

    np.savez_compressed(
        temp_file,
        node_frames=node_frames,
        edge_flow_frames=edge_frames,
        window_end_frame=end_frames,
        y_horizons=y,
        onset_class=onset,
        family_class=family,
        episode_id=episodes,
        minute=minutes,
        node_feature_names=np.asarray(
            train_manifest["node_feature_names"]
        )
    )

    temp_file.replace(GRAPH_FILE)

    validation_manifest = {
        "version": "VALIDATION_only_v1",
        "source_validation_sha256": source_sha,
        "source_train_graph_sha256": train_manifest["artifact_sha256"],
        "artifact_sha256": file_sha256(GRAPH_FILE),
        "generator_seed": SEED,
        "generator": "SAME_AS_TRAIN_only_v1",
        "windows": int(len(X)),
        "episodes": int(episode_count),
        "frames": int(cursor),
        "nodes_per_graph": 8,
        "node_features": 8,
        "history_minutes": 10,
        "forecast_horizons": [5, 10, 15],
        "graph_origin": "SYNTHETIC_FROM_AGGREGATE_FEATURES",
        "scaler_fitted_on_validation": False,
        "official_test_accessed": False
    }

    MANIFEST_FILE.write_text(
        json.dumps(validation_manifest, indent=2)
    )

    print("\n" + "=" * 85)
    print("GRAPH COMPANION VALIDATION BUILD: PASS")
    print("=" * 85)

    print("Validation windows:", len(X))
    print("Validation episodes:", episode_count)
    print("Graph frames:", cursor)
    print("Node tensor:", node_frames.shape)
    print("Edge tensor:", edge_frames.shape)
    print("\nSaved:", GRAPH_FILE)
    print("SHA256:", file_sha256(GRAPH_FILE))

print("\nTRAIN checkpoints modified: NO")
print("Frozen V2 data modified: NO")
print("Official TEST accessed: NO")
print("Model evaluation: NEXT STEP")



In [ ]:

# =============================================================================
# CELL 5 — NODE-AWARE LSTM + GRAPH LSTM VALIDATION
#
# Loads both already-trained checkpoints.
# Uses each model's original TRAIN-fitted scaler.
# Evaluates on frozen V2 VALIDATION graph companion.
#
# METRICS: AP, AUROC, Brier, Precision, Recall, F1,
#          false-positive rate, benign-episode false-alert rate.
#
# NO RETRAINING
# NO MODEL OVERWRITING
# NO TEST ACCESS
# READ ONLY
# =============================================================================

from pathlib import Path
import hashlib
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# =============================================================================
# 1. PATHS AND SETTINGS
# =============================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

GRAPH_ROOT = ROOT / "Synthetic_NAF_v2_GraphCompanion"

TRAIN_MANIFEST = (
    GRAPH_ROOT / "TRAIN_only_v1" /
    "GRAPH_COMPANION_MANIFEST.json"
)

VAL_DIR = GRAPH_ROOT / "VALIDATION_only_v1"

VAL_FILE = VAL_DIR / "VALIDATION_temporal_graphs.npz"

VAL_MANIFEST = (
    VAL_DIR / "GRAPH_COMPANION_VALIDATION_MANIFEST.json"
)

NODE_MODEL_FILE = (
    GRAPH_ROOT / "models" /
    "NodeAware_LSTM" /
    "train_only_v1" /
    "NODE_LSTM_STATE.pt"
)

GRAPH_MODEL_FILE = (
    GRAPH_ROOT / "models" /
    "Graph_LSTM" /
    "train_only_v1" /
    "GRAPH_LSTM_STATE.pt"
)

ORIGINAL_VAL = (
    ROOT / "Synthetic_NAF_v2" /
    "release_candidate_corrected" /
    "windows_W10_VALIDATION.npz"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 256

# Diagnostic threshold only.
# NOT a calibrated or optimized deployment threshold.
THRESHOLD = 0.50

HORIZONS = [5, 10, 15]


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


print("=" * 90)
print("CELL 5 — GRAPH MODEL VALIDATION")
print("=" * 90)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

# =============================================================================
# 2. VERIFY ALL REQUIRED FILES
# =============================================================================

required_files = [
    TRAIN_MANIFEST,
    VAL_FILE,
    VAL_MANIFEST,
    NODE_MODEL_FILE,
    GRAPH_MODEL_FILE,
    ORIGINAL_VAL
]

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(path)

train_manifest = json.loads(
    TRAIN_MANIFEST.read_text(encoding="utf-8")
)

val_manifest = json.loads(
    VAL_MANIFEST.read_text(encoding="utf-8")
)

val_sha = sha256(VAL_FILE)

assert val_sha == val_manifest["artifact_sha256"]

assert (
    val_manifest["source_train_graph_sha256"]
    == train_manifest["artifact_sha256"]
)

assert val_manifest["generator_seed"] == train_manifest["seed"]

assert val_manifest["nodes_per_graph"] == 8
assert val_manifest["history_minutes"] == 10
assert val_manifest["forecast_horizons"] == [5, 10, 15]

print("\nVALIDATION graph integrity: PASS")
print("Graph TRAIN / VALIDATION generator consistency: PASS")

# =============================================================================
# 3. LOAD VALIDATION GRAPH ARRAYS
# =============================================================================

with np.load(VAL_FILE, allow_pickle=False) as data:

    node_frames = data["node_frames"].astype(np.float32)

    edge_frames = data["edge_flow_frames"].astype(np.float32)

    end_frames = data["window_end_frame"].astype(np.int64)

    y_true = data["y_horizons"].astype(np.int8)

    episodes = data["episode_id"].astype(str)

    minutes = data["minute"].astype(np.int32)

assert node_frames.shape[1:] == (8, 8)
assert edge_frames.shape == (len(node_frames), 8, 8)

assert y_true.shape == (len(end_frames), 3)
assert len(episodes) == len(minutes) == len(end_frames)

assert end_frames.min() >= 9
assert end_frames.max() < len(node_frames)

assert np.isfinite(node_frames).all()
assert np.isfinite(edge_frames).all()

assert np.isin(y_true, [0, 1]).all()

assert np.all(y_true[:, 0] <= y_true[:, 1])
assert np.all(y_true[:, 1] <= y_true[:, 2])

# Confirm row alignment with the actual frozen validation labels.
with np.load(ORIGINAL_VAL, allow_pickle=False) as data:

    original_y = data["y_horizons"].astype(np.int8)

    original_episodes = data["episode_id"].astype(str)

    original_minutes = data["minute"].astype(np.int32)

assert np.array_equal(y_true, original_y)
assert np.array_equal(episodes, original_episodes)
assert np.array_equal(minutes, original_minutes)

print("Frozen validation label alignment: PASS")

print("\nVALIDATION DATA")
print("Windows:", len(end_frames))
print("Episodes:", len(np.unique(episodes)))
print("Graph snapshots:", len(node_frames))
print("Node features:", node_frames.shape)
print("Edges:", edge_frames.shape)

for j, h in enumerate(HORIZONS):
    print(
        f"H{h} positive prevalence:",
        f"{y_true[:, j].mean():.4%}"
    )

# =============================================================================
# 4. RECREATE ORIGINAL NODE-AWARE LSTM ARCHITECTURE
# =============================================================================

class NodeAwareLSTM(nn.Module):

    def __init__(self, input_dim=8, hidden_dim=32):

        super().__init__()

        self.node_lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.attention = nn.Linear(hidden_dim, 1)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 3)
        )

    def forward(self, x):

        b, t, n, f = x.shape

        x = x.permute(0, 2, 1, 3)
        x = x.reshape(b * n, t, f)

        _, (hidden, _) = self.node_lstm(x)

        node_embeddings = hidden[-1].reshape(b, n, -1)

        weights = torch.softmax(
            self.attention(node_embeddings),
            dim=1
        )

        network_embedding = (
            node_embeddings * weights
        ).sum(dim=1)

        logits = self.head(network_embedding)

        hazards = torch.sigmoid(logits)

        probabilities = 1.0 - torch.cumprod(
            1.0 - hazards,
            dim=1
        )

        return probabilities.clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )


# =============================================================================
# 5. RECREATE ORIGINAL GRAPH + LSTM ARCHITECTURE
# =============================================================================

class WeightedGraphConv(nn.Module):

    def __init__(self, in_dim, out_dim):

        super().__init__()

        self.self_linear = nn.Linear(
            in_dim, out_dim
        )

        self.neighbor_linear = nn.Linear(
            in_dim, out_dim
        )

    def forward(self, x, adjacency):

        incoming = torch.bmm(
            adjacency, x
        )

        result = (
            self.self_linear(x)
            + self.neighbor_linear(incoming)
        )

        return torch.relu(result)


class GraphLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_dim=8,
        graph_hidden=32,
        lstm_hidden=64,
        dropout=0.1
    ):

        super().__init__()

        self.graph_conv1 = WeightedGraphConv(
            input_dim,
            graph_hidden
        )

        self.graph_conv2 = WeightedGraphConv(
            graph_hidden,
            graph_hidden
        )

        self.graph_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=graph_hidden * 2,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    def forward(self, nodes, edges):

        b, t, n, f = nodes.shape

        x = nodes.reshape(b * t, n, f)

        adjacency = edges.reshape(b * t, n, n)

        # Training used edge matrices [source, target].
        # Incoming message aggregation uses [target, source].
        adjacency = adjacency.transpose(1, 2)

        degree = adjacency.sum(
            dim=-1,
            keepdim=True
        )

        adjacency = adjacency / degree.clamp(
            min=1e-8
        )

        h = self.graph_conv1(x, adjacency)

        h = self.graph_dropout(h)

        h = self.graph_conv2(h, adjacency)

        mean_pool = h.mean(dim=1)

        max_pool = h.max(dim=1).values

        graph_embedding = torch.cat(
            [mean_pool, max_pool],
            dim=-1
        )

        graph_sequence = graph_embedding.reshape(
            b, t, -1
        )

        temporal_output, _ = self.lstm(
            graph_sequence
        )

        last_state = temporal_output[:, -1, :]

        logits = self.head(last_state)

        hazards = torch.sigmoid(logits)

        probabilities = 1.0 - torch.cumprod(
            1.0 - hazards,
            dim=1
        )

        return probabilities.clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )


# =============================================================================
# 6. LOAD ORIGINAL CHECKPOINTS AND TRAIN-FITTED SCALERS
# =============================================================================

print("\nLoading trained checkpoints...")

# Only load your own trusted checkpoints from Drive.
node_checkpoint = torch.load(
    NODE_MODEL_FILE,
    map_location="cpu",
    weights_only=False
)

graph_checkpoint = torch.load(
    GRAPH_MODEL_FILE,
    map_location="cpu",
    weights_only=False
)

expected_graph_sha = train_manifest["artifact_sha256"]

assert node_checkpoint["graph_sha256"] == expected_graph_sha

assert graph_checkpoint["graph_sha256"] == expected_graph_sha

node_model = NodeAwareLSTM(
    input_dim=8,
    hidden_dim=32
).to(DEVICE)

node_model.load_state_dict(
    node_checkpoint["model_state"],
    strict=True
)

cfg = graph_checkpoint["config"]

graph_model = GraphLSTMForecaster(
    input_dim=cfg["input_features"],
    graph_hidden=cfg["graph_hidden"],
    lstm_hidden=cfg["lstm_hidden"],
    dropout=cfg["dropout"]
).to(DEVICE)

graph_model.load_state_dict(
    graph_checkpoint["model_state"],
    strict=True
)

node_model.eval()
graph_model.eval()

print("Node-aware checkpoint: PASS")
print("Graph + LSTM checkpoint: PASS")

print("Node-aware SHA256:", sha256(NODE_MODEL_FILE))
print("Graph + LSTM SHA256:", sha256(GRAPH_MODEL_FILE))

# =============================================================================
# 7. APPLY ORIGINAL TRAIN-ONLY SCALERS
# =============================================================================

def transform_nodes(raw_frames, checkpoint):

    median = np.asarray(
        checkpoint["median"],
        dtype=np.float32
    )

    scale = np.asarray(
        checkpoint["scale"],
        dtype=np.float32
    )

    assert median.shape == (8,)
    assert scale.shape == (8,)
    assert np.isfinite(median).all()
    assert np.isfinite(scale).all()
    assert (scale > 0).all()

    transformed = np.log1p(raw_frames)

    transformed = np.clip(
        (transformed - median) / scale,
        -8.0,
        8.0
    ).astype(np.float32)

    assert np.isfinite(transformed).all()

    return transformed


node_input = transform_nodes(
    node_frames,
    node_checkpoint
)

graph_node_input = transform_nodes(
    node_frames,
    graph_checkpoint
)

# Identical edge normalization to Graph + LSTM training
edge_total = edge_frames.sum(
    axis=(1, 2),
    keepdims=True
)

graph_edge_input = (
    edge_frames / np.maximum(edge_total, 1e-8)
).astype(np.float32)

assert np.isfinite(graph_edge_input).all()

print("\nOriginal TRAIN-fitted preprocessing: PASS")
print("Scaler refitted on validation: NO")

# =============================================================================
# 8. GENERATE PREDICTIONS
# =============================================================================

def predict_model(model, frames, edge_data=None):

    predictions = []

    offsets = np.arange(-9, 1, dtype=np.int64)

    model.eval()

    started = time.time()

    with torch.inference_mode():

        for start in range(0, len(end_frames), BATCH_SIZE):

            stop = min(
                start + BATCH_SIZE,
                len(end_frames)
            )

            ids = np.arange(start, stop)

            indices = (
                end_frames[ids, None]
                + offsets[None, :]
            )

            x = torch.from_numpy(
                frames[indices]
            ).to(DEVICE)

            if edge_data is None:

                p = model(x)

            else:

                e = torch.from_numpy(
                    edge_data[indices]
                ).to(DEVICE)

                p = model(x, e)

            predictions.append(
                p.cpu().numpy()
            )

            if stop % 5000 < BATCH_SIZE or stop == len(end_frames):
                print(
                    f"  Predicted {stop:,}/{len(end_frames):,} windows",
                    flush=True
                )

    result = np.concatenate(
        predictions,
        axis=0
    ).astype(np.float64)

    assert result.shape == (len(end_frames), 3)

    assert np.isfinite(result).all()

    assert (result >= 0).all()
    assert (result <= 1).all()

    assert np.all(
        result[:, 0] <= result[:, 1] + 1e-7
    )

    assert np.all(
        result[:, 1] <= result[:, 2] + 1e-7
    )

    print(
        "  Inference seconds:",
        round(time.time() - started, 2)
    )

    return result


print("\n" + "=" * 85)
print("NODE-AWARE LSTM INFERENCE")
print("=" * 85)

node_probs = predict_model(
    node_model,
    node_input
)

print("\n" + "=" * 85)
print("GRAPH + LSTM INFERENCE")
print("=" * 85)

graph_probs = predict_model(
    graph_model,
    graph_node_input,
    graph_edge_input
)

print("\nPrediction shapes:", node_probs.shape, graph_probs.shape)
print("Prediction monotonicity: PASS")

# =============================================================================
# 9. COMPUTE EVALUATION METRICS
# =============================================================================

def evaluate_model(name, probabilities):

    results = []

    for j, horizon in enumerate(HORIZONS):

        y = y_true[:, j].astype(int)

        p = probabilities[:, j]

        predicted = (p >= THRESHOLD).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y,
            predicted,
            labels=[0, 1]
        ).ravel()

        ap = average_precision_score(y, p)

        auc = (
            roc_auc_score(y, p)
            if len(np.unique(y)) == 2
            else np.nan
        )

        brier = brier_score_loss(y, p)

        precision = precision_score(
            y, predicted, zero_division=0
        )

        recall = recall_score(
            y, predicted, zero_division=0
        )

        f1 = f1_score(
            y, predicted, zero_division=0
        )

        false_positive_rate = (
            fp / (fp + tn)
            if (fp + tn) > 0
            else np.nan
        )

        results.append({
            "Model": name,
            "Horizon": f"H{horizon}",
            "AP": ap,
            "AUROC": auc,
            "Brier": brier,
            "Precision@0.5": precision,
            "Recall@0.5": recall,
            "F1@0.5": f1,
            "FPR@0.5": false_positive_rate,
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn)
        })

    return results


metrics = pd.DataFrame(
    evaluate_model("Node-aware LSTM", node_probs)
    + evaluate_model("Graph + LSTM", graph_probs)
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

print("\n" + "=" * 90)
print("MODEL VALIDATION RESULTS")
print("=" * 90)

print(
    metrics[
        [
            "Model",
            "Horizon",
            "AP",
            "AUROC",
            "Brier",
            "Precision@0.5",
            "Recall@0.5",
            "F1@0.5",
            "FPR@0.5"
        ]
    ].round(6).to_string(index=False)
)

# =============================================================================
# 10. FALSE ALERTS AT THE EPISODE LEVEL
# =============================================================================

print("\n" + "=" * 90)
print("BENIGN EPISODE FALSE-ALERT ANALYSIS")
print("=" * 90)

for j, horizon in enumerate(HORIZONS):

    # An episode is 'benign for this horizon' when none
    # of its prediction windows has a positive H label.

    grouped = pd.DataFrame({
        "episode": episodes,
        "label": y_true[:, j],
        "node_p": node_probs[:, j],
        "graph_p": graph_probs[:, j]
    }).groupby("episode").agg(
        label_max=("label", "max"),
        node_max=("node_p", "max"),
        graph_max=("graph_p", "max")
    )

    benign = grouped[grouped["label_max"] == 0]

    if len(benign) == 0:
        print(f"H{horizon}: No wholly negative episodes")
        continue

    node_false_rate = (
        benign["node_max"] >= THRESHOLD
    ).mean()

    graph_false_rate = (
        benign["graph_max"] >= THRESHOLD
    ).mean()

    print(
        f"H{horizon} | "
        f"Benign episodes={len(benign)} | "
        f"Node false-alert episodes={node_false_rate:.2%} | "
        f"Graph false-alert episodes={graph_false_rate:.2%}"
    )

# =============================================================================
# 11. COMPARISON WITH EXISTING V2 BASELINES
# =============================================================================

historical = pd.DataFrame([
    {
        "Model": "XGBoost",
        "H5 AP": 0.676304,
        "H10 AP": 0.875130,
        "H15 AP": np.nan
    },
    {
        "Model": "Single-layer LSTM",
        "H5 AP": 0.713229,
        "H10 AP": 0.896284,
        "H15 AP": 0.926021
    },
    {
        "Model": "Stacked LSTM",
        "H5 AP": 0.761005,
        "H10 AP": 0.906018,
        "H15 AP": 0.935854
    }
])

new_rows = []

for model_name in ["Node-aware LSTM", "Graph + LSTM"]:

    part = metrics[metrics["Model"] == model_name]

    new_rows.append({
        "Model": model_name,
        "H5 AP": float(
            part.loc[part["Horizon"] == "H5", "AP"].iloc[0]
        ),
        "H10 AP": float(
            part.loc[part["Horizon"] == "H10", "AP"].iloc[0]
        ),
        "H15 AP": float(
            part.loc[part["Horizon"] == "H15", "AP"].iloc[0]
        )
    })

comparison = pd.concat(
    [historical, pd.DataFrame(new_rows)],
    ignore_index=True
)

print("\n" + "=" * 90)
print("FIVE-MODEL VALIDATION AP COMPARISON")
print("=" * 90)

print(
    comparison.round(6).to_string(index=False)
)

print(
    "\nNOTE: Existing baseline scores are previously "
    "reported V2 validation values, not newly replayed "
    "in this cell."
)

print(
    "Graph models use simulated node/edge structure "
    "derived from aggregate V2 features."
)

# =============================================================================
# 12. FINAL INTEGRITY REPORT
# =============================================================================

print("\n" + "=" * 90)
print("GRAPH MODEL VALIDATION COMPLETE")
print("=" * 90)

print("Node-aware LSTM: EVALUATED")
print("Graph + LSTM: EVALUATED")
print("Validation windows:", len(end_frames))
print("Validation episodes:", len(np.unique(episodes)))

print("Original TRAIN checkpoints modified: NO")
print("Validation scaler fitting performed: NO")
print("Frozen V2 data modified: NO")
print("Official TEST accessed: NO")
print("Results written to Drive: NO")

print("=" * 90)



In [ ]:

# =============================================================================
# CELL 6 — GRAPH + LSTM HYPERPARAMETER OPTIMIZATION
#
# FAST TRAIN-ONLY EPISODE-GROUPED CROSS-VALIDATION
#
# 4 configurations
# 2 episode-stratified folds
# Maximum 8 epochs per fold, with early stopping
#
# Selection score:
# 60% H5 AP + 30% H10 AP + 10% H15 AP
#
# IMPORTANT:
# - Uses frozen GraphCompanion TRAIN only
# - Fits a separate scaler within each TRAIN fold
# - Never mixes episodes across training and holdout folds
# - Saves checkpoints/progress separately from original model
# - No official VALIDATION or TEST access
# =============================================================================

from pathlib import Path
import hashlib
import json
import time
import os
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score

from google.colab import drive

# =============================================================================
# 1. ENVIRONMENT
# =============================================================================

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

GRAPH_ROOT = ROOT / "Synthetic_NAF_v2_GraphCompanion"

GRAPH_FILE = (
    GRAPH_ROOT
    / "TRAIN_only_v1"
    / "TRAIN_temporal_graphs.npz"
)

MANIFEST_FILE = (
    GRAPH_ROOT
    / "TRAIN_only_v1"
    / "GRAPH_COMPANION_MANIFEST.json"
)

# New output folder. Original model is untouched.
SEARCH_DIR = (
    GRAPH_ROOT
    / "models"
    / "Graph_LSTM"
    / "cv_search_v1"
)

SEED = 20261027

N_FOLDS = 2
MAX_EPOCHS = 8
PATIENCE = 3

BATCH_SIZE = 256
WEIGHT_DECAY = 0.001

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if DEVICE.type == "cpu":
    torch.set_num_threads(
        min(4, os.cpu_count() or 1)
    )

CONFIGS = [
    {
        "name": "C01_baseline",
        "graph_hidden": 32,
        "lstm_hidden": 64,
        "dropout": 0.10,
        "lr": 0.001
    },
    {
        "name": "C02_wider_graph",
        "graph_hidden": 64,
        "lstm_hidden": 64,
        "dropout": 0.10,
        "lr": 0.001
    },
    {
        "name": "C03_more_regularized",
        "graph_hidden": 32,
        "lstm_hidden": 64,
        "dropout": 0.20,
        "lr": 0.0005
    },
    {
        "name": "C04_wider_lstm",
        "graph_hidden": 32,
        "lstm_hidden": 96,
        "dropout": 0.10,
        "lr": 0.0005
    }
]


def file_sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def atomic_json(data, path):
    temp = path.with_name(path.name + ".tmp")

    temp.write_text(
        json.dumps(data, indent=2),
        encoding="utf-8"
    )

    temp.replace(path)


def atomic_torch(data, path):
    temp = path.with_name(path.name + ".tmp")

    torch.save(data, temp)

    temp.replace(path)


print("=" * 85)
print("CELL 6 — GRAPH + LSTM TRAIN-ONLY OPTIMIZATION")
print("=" * 85)

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Configurations:", len(CONFIGS))
print("Grouped folds:", N_FOLDS)
print("Maximum epochs:", MAX_EPOCHS)

# =============================================================================
# 2. VERIFY TRAIN GRAPH DATA
# =============================================================================

for path in [GRAPH_FILE, MANIFEST_FILE]:
    if not path.is_file():
        raise FileNotFoundError(path)

manifest = json.loads(
    MANIFEST_FILE.read_text(encoding="utf-8")
)

graph_sha = file_sha256(GRAPH_FILE)

assert graph_sha == manifest["artifact_sha256"]
assert manifest["nodes_per_graph"] == 8
assert manifest["history_minutes"] == 10
assert manifest["forecast_horizons_minutes"] == [5, 10, 15]

with np.load(GRAPH_FILE, allow_pickle=False) as pack:

    nodes_raw = pack["node_frames"].astype(np.float32)

    edges_raw = pack["edge_flow_frames"].astype(np.float32)

    end_frames = pack["window_end_frame"].astype(np.int64)

    y_all = pack["y_horizons"].astype(np.float32)

    episode_ids = pack["episode_id"].astype(str)

assert nodes_raw.ndim == 3
assert nodes_raw.shape[1:] == (8, 8)

assert edges_raw.shape == (
    len(nodes_raw), 8, 8
)

assert y_all.shape == (len(end_frames), 3)

assert len(episode_ids) == len(end_frames)

assert end_frames.min() >= 9
assert end_frames.max() < len(nodes_raw)

assert np.isfinite(nodes_raw).all()
assert np.isfinite(edges_raw).all()

assert (nodes_raw >= 0).all()
assert (edges_raw >= 0).all()

assert np.isin(y_all, [0, 1]).all()

assert np.all(y_all[:, 0] <= y_all[:, 1])
assert np.all(y_all[:, 1] <= y_all[:, 2])

print("\nTRAIN graph integrity: PASS")
print("Training windows:", len(end_frames))
print("Graph frames:", len(nodes_raw))
print("Unique episodes:", len(np.unique(episode_ids)))

# =============================================================================
# 3. CREATE EPISODE-GROUPED CROSS-VALIDATION SPLITS
# =============================================================================

unique_eps, group_id = np.unique(
    episode_ids,
    return_inverse=True
)

# Stratify groups using whether an episode contains
# at least one H5-positive training window.
ep_h5 = np.zeros(
    len(unique_eps),
    dtype=np.int8
)

np.maximum.at(
    ep_h5,
    group_id,
    y_all[:, 0].astype(np.int8)
)

if (
    len(np.unique(ep_h5)) != 2
    or min(np.bincount(ep_h5)) < N_FOLDS
):
    raise RuntimeError(
        "Not enough positive and negative episodes."
    )

splitter = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

folds = []

for fold_num, (train_groups, val_groups) in enumerate(
    splitter.split(
        np.arange(len(unique_eps)),
        ep_h5
    ),
    start=1
):

    train_mask = np.isin(
        group_id,
        train_groups
    )

    val_mask = np.isin(
        group_id,
        val_groups
    )

    train_idx = np.flatnonzero(train_mask)
    val_idx = np.flatnonzero(val_mask)

    assert set(
        episode_ids[train_idx]
    ).isdisjoint(
        set(episode_ids[val_idx])
    )

    for j in range(3):
        if np.unique(
            y_all[val_idx, j]
        ).size != 2:
            raise RuntimeError(
                f"Fold {fold_num} lacks both label "
                f"classes for H{[5, 10, 15][j]}"
            )

    folds.append(
        (train_idx, val_idx)
    )

    print(
        f"\nFold {fold_num}: "
        f"TRAIN={len(train_idx):,} windows, "
        f"HOLDOUT={len(val_idx):,} windows"
    )

    print(
        "TRAIN episodes:",
        len(train_groups),
        "| HOLDOUT episodes:",
        len(val_groups)
    )

    print(
        "HOLDOUT H5 positives:",
        int(y_all[val_idx, 0].sum())
    )

print("\nEpisode leakage check: PASS")

# =============================================================================
# 4. EDGE PREPROCESSING
# =============================================================================

# Edge fractions do not require any learned scaling.
# The original Graph + LSTM uses this same transformation.

edge_totals = edges_raw.sum(
    axis=(1, 2),
    keepdims=True
)

edges_scaled = (
    edges_raw
    / np.maximum(edge_totals, 1e-8)
).astype(np.float32)

del edges_raw, edge_totals

assert np.isfinite(edges_scaled).all()

# =============================================================================
# 5. ORIGINAL GRAPH CONVOLUTION ARCHITECTURE
# =============================================================================

class WeightedGraphConv(nn.Module):

    def __init__(self, in_dim, out_dim):
        super().__init__()

        self.self_linear = nn.Linear(
            in_dim,
            out_dim
        )

        self.neighbor_linear = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(self, x, adjacency):

        incoming = torch.bmm(
            adjacency,
            x
        )

        return torch.relu(
            self.self_linear(x)
            + self.neighbor_linear(incoming)
        )


class GraphLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_dim=8,
        graph_hidden=32,
        lstm_hidden=64,
        dropout=0.1
    ):
        super().__init__()

        self.graph_conv1 = WeightedGraphConv(
            input_dim,
            graph_hidden
        )

        self.graph_conv2 = WeightedGraphConv(
            graph_hidden,
            graph_hidden
        )

        self.graph_dropout = nn.Dropout(
            dropout
        )

        self.lstm = nn.LSTM(
            input_size=graph_hidden * 2,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    def forward(self, nodes, edges):

        b, t, n, f = nodes.shape

        x = nodes.reshape(
            b * t, n, f
        )

        adjacency = edges.reshape(
            b * t, n, n
        )

        adjacency = adjacency.transpose(1, 2)

        degree = adjacency.sum(
            dim=-1,
            keepdim=True
        )

        adjacency = adjacency / degree.clamp(
            min=1e-8
        )

        h = self.graph_conv1(
            x, adjacency
        )

        h = self.graph_dropout(h)

        h = self.graph_conv2(
            h, adjacency
        )

        pooled = torch.cat(
            [
                h.mean(dim=1),
                h.max(dim=1).values
            ],
            dim=-1
        )

        sequence = pooled.reshape(
            b, t, -1
        )

        temporal, _ = self.lstm(
            sequence
        )

        hazards = torch.sigmoid(
            self.head(
                temporal[:, -1, :]
            )
        )

        return (
            1.0 - torch.cumprod(
                1.0 - hazards,
                dim=1
            )
        ).clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )


# =============================================================================
# 6. SAVE AND VERIFY OPTIMIZATION PROTOCOL
# =============================================================================

SEARCH_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RUN_META = {
    "dataset_sha256": graph_sha,
    "search_version": "v1",
    "seed": SEED,
    "n_folds": N_FOLDS,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "batch_size": BATCH_SIZE,
    "weight_decay": WEIGHT_DECAY,
    "selection": (
        "mean fold best "
        "(0.6*AP_H5+0.3*AP_H10+0.1*AP_H15)"
    ),
    "configs": CONFIGS
}

meta_path = SEARCH_DIR / "CV_SEARCH_PROTOCOL.json"

if meta_path.exists():

    old_meta = json.loads(
        meta_path.read_text()
    )

    if old_meta != RUN_META:
        raise RuntimeError(
            "Existing CV protocol differs. "
            "Use a new output folder."
        )

else:

    atomic_json(
        RUN_META,
        meta_path
    )

# =============================================================================
# 7. TRAIN-ONLY CROSS-VALIDATION SEARCH
# =============================================================================

frame_offsets = np.arange(
    -9, 1,
    dtype=np.int64
)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    folds,
    start=1
):

    # ============================================================
    # FIT SCALER EXCLUSIVELY ON FOLD-TRAIN FRAMES
    # ============================================================

    train_frame_ids = np.unique(
        (
            end_frames[train_idx, None]
            + frame_offsets
        ).ravel()
    )

    val_frame_ids = np.unique(
        (
            end_frames[val_idx, None]
            + frame_offsets
        ).ravel()
    )

    if np.intersect1d(
        train_frame_ids,
        val_frame_ids
    ).size:
        raise RuntimeError(
            "Graph frames overlap across folds."
        )

    flat_train = np.log1p(
        nodes_raw[train_frame_ids]
    ).reshape(-1, 8)

    median = np.median(
        flat_train,
        axis=0
    ).astype(np.float32)

    q25, q75 = np.percentile(
        flat_train,
        [25, 75],
        axis=0
    )

    scale = np.maximum(
        q75 - q25,
        0.05
    ).astype(np.float32)

    del flat_train

    nodes_scaled = np.clip(
        (np.log1p(nodes_raw) - median) / scale,
        -8.0,
        8.0
    ).astype(np.float32)

    print("\n" + "=" * 85)
    print(f"FOLD {fold}/{N_FOLDS}")
    print("=" * 85)

    print("Scaler fitted using fold TRAIN only: YES")
    print("Train/holdout graph overlap: NONE")

    # ============================================================
    # TRY FOUR CONFIGURATIONS
    # ============================================================

    for config_index, cfg in enumerate(CONFIGS):

        tag = f"{cfg['name']}_fold{fold}"

        job_dir = SEARCH_DIR / tag

        job_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        result_path = job_dir / "RESULT.json"
        progress_path = job_dir / "PROGRESS.pt"
        best_path = job_dir / "BEST_STATE.pt"

        # --------------------------------------------------------
        # Skip already completed and verified configurations
        # --------------------------------------------------------

        if result_path.exists():

            result = json.loads(
                result_path.read_text()
            )

            if (
                result.get("graph_sha256") != graph_sha
                or result.get("config") != cfg
            ):
                raise RuntimeError(
                    f"Existing result mismatch: {tag}"
                )

            if (
                not best_path.is_file()
                or file_sha256(best_path)
                != result["best_sha256"]
            ):
                raise RuntimeError(
                    f"Best checkpoint missing or changed: {tag}"
                )

            print(
                f"{tag}: ALREADY COMPLETE "
                f"| score={result['best_score']:.5f}"
            )

            fold_results.append(result)

            continue

        # --------------------------------------------------------
        # Initialize separate model for this configuration
        # --------------------------------------------------------

        run_seed = (
            SEED
            + 10000 * fold
            + 100 * config_index
        )

        random.seed(run_seed)
        np.random.seed(run_seed)
        torch.manual_seed(run_seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(run_seed)

        model = GraphLSTMForecaster(
            input_dim=8,
            graph_hidden=cfg["graph_hidden"],
            lstm_hidden=cfg["lstm_hidden"],
            dropout=cfg["dropout"]
        ).to(DEVICE)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg["lr"],
            weight_decay=WEIGHT_DECAY
        )

        loss_fn = nn.BCELoss()

        epoch_start = 0
        best_score = -1.0
        best_epoch = 0
        bad_epochs = 0

        best_aps = None
        history = []

        # --------------------------------------------------------
        # Resume progress after runtime disconnection
        # --------------------------------------------------------

        if progress_path.exists():

            state = torch.load(
                progress_path,
                map_location="cpu",
                weights_only=False
            )

            if (
                state["graph_sha256"] != graph_sha
                or state["config"] != cfg
                or state["run_seed"] != run_seed
            ):
                raise RuntimeError(
                    f"Progress checkpoint mismatch: {tag}"
                )

            model.load_state_dict(
                state["model_state"]
            )

            optimizer.load_state_dict(
                state["optimizer_state"]
            )

            epoch_start = int(state["epoch"])
            best_score = float(state["best_score"])
            best_epoch = int(state["best_epoch"])
            bad_epochs = int(state["bad_epochs"])

            best_aps = state["best_aps"]
            history = state["history"]

            torch.set_rng_state(
                state["torch_rng_state"]
            )

            if (
                DEVICE.type == "cuda"
                and "cuda_rng_state" in state
            ):
                torch.cuda.set_rng_state_all(
                    state["cuda_rng_state"]
                )

            if best_epoch and not best_path.exists():
                raise RuntimeError(
                    f"Missing saved best state: {tag}"
                )

            print(
                f"{tag}: Resuming at epoch "
                f"{epoch_start + 1}"
            )

        # ========================================================
        # TRAIN AND EVALUATE EACH EPOCH
        # ========================================================

        for epoch in range(
            epoch_start,
            MAX_EPOCHS
        ):

            started = time.monotonic()

            model.train()

            epoch_order = np.random.default_rng(
                run_seed + epoch
            ).permutation(train_idx)

            cumulative_loss = 0.0
            samples = 0

            for p in range(
                0,
                len(epoch_order),
                BATCH_SIZE
            ):

                ids = epoch_order[
                    p:p + BATCH_SIZE
                ]

                frame_ids = (
                    end_frames[ids, None]
                    + frame_offsets
                )

                x = torch.from_numpy(
                    nodes_scaled[frame_ids]
                ).to(DEVICE)

                e = torch.from_numpy(
                    edges_scaled[frame_ids]
                ).to(DEVICE)

                y = torch.from_numpy(
                    y_all[ids]
                ).to(DEVICE)

                optimizer.zero_grad(
                    set_to_none=True
                )

                pred = model(x, e)

                loss = loss_fn(
                    pred, y
                )

                if not torch.isfinite(loss):
                    raise RuntimeError(
                        f"Non-finite training loss: "
                        f"{tag} epoch {epoch + 1}"
                    )

                loss.backward()

                nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0
                )

                optimizer.step()

                cumulative_loss += (
                    float(loss.item()) * len(ids)
                )

                samples += len(ids)

            # ====================================================
            # EVALUATE ON HELD-OUT TRAIN EPISODES
            # ====================================================

            model.eval()

            preds = []

            with torch.inference_mode():

                for p in range(
                    0,
                    len(val_idx),
                    BATCH_SIZE
                ):

                    ids = val_idx[
                        p:p + BATCH_SIZE
                    ]

                    frame_ids = (
                        end_frames[ids, None]
                        + frame_offsets
                    )

                    x = torch.from_numpy(
                        nodes_scaled[frame_ids]
                    ).to(DEVICE)

                    e = torch.from_numpy(
                        edges_scaled[frame_ids]
                    ).to(DEVICE)

                    preds.append(
                        model(x, e).cpu().numpy()
                    )

            pred = np.concatenate(
                preds,
                axis=0
            )

            aps = [
                float(
                    average_precision_score(
                        y_all[val_idx, j],
                        pred[:, j]
                    )
                )
                for j in range(3)
            ]

            # H5 carries the greatest importance.
            score = (
                0.6 * aps[0]
                + 0.3 * aps[1]
                + 0.1 * aps[2]
            )

            history.append({
                "epoch": epoch + 1,
                "train_bce": cumulative_loss / samples,
                "ap_h5": aps[0],
                "ap_h10": aps[1],
                "ap_h15": aps[2],
                "score": score
            })

            # ====================================================
            # SAVE BEST FOLD MODEL
            # ====================================================

            if score > best_score + 1e-4:

                best_score = score
                best_epoch = epoch + 1
                best_aps = aps
                bad_epochs = 0

                atomic_torch(
                    {
                        "model_state": model.state_dict(),
                        "median": median,
                        "scale": scale,
                        "config": cfg,
                        "graph_sha256": graph_sha,
                        "fold": fold,
                        "best_epoch": best_epoch
                    },
                    best_path
                )

            else:

                bad_epochs += 1

            # ====================================================
            # SAVE PROGRESS AFTER EACH EPOCH
            # ====================================================

            state = {
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "graph_sha256": graph_sha,
                "config": cfg,
                "run_seed": run_seed,
                "epoch": epoch + 1,
                "best_score": best_score,
                "best_epoch": best_epoch,
                "best_aps": best_aps,
                "bad_epochs": bad_epochs,
                "history": history,
                "torch_rng_state": torch.get_rng_state()
            }

            if DEVICE.type == "cuda":

                state["cuda_rng_state"] = (
                    torch.cuda.get_rng_state_all()
                )

            atomic_torch(
                state,
                progress_path
            )

            elapsed = time.monotonic() - started

            print(
                f"{tag} "
                f"Epoch {epoch + 1:02d}/{MAX_EPOCHS}"
                f" | BCE={history[-1]['train_bce']:.5f}"
                f" | H5={aps[0]:.5f}"
                f" | H10={aps[1]:.5f}"
                f" | H15={aps[2]:.5f}"
                f" | SCORE={score:.5f}"
                f" | seconds={elapsed:.1f}",
                flush=True
            )

            # ====================================================
            # EARLY STOPPING
            # ====================================================

            if (
                bad_epochs >= PATIENCE
                and epoch + 1 >= 4
            ):

                print(
                    f"{tag}: Early stopping. "
                    f"Best epoch={best_epoch}",
                    flush=True
                )

                break

        # ========================================================
        # SAVE CONFIGURATION/FOLD RESULT
        # ========================================================

        result = {
            "tag": tag,
            "fold": fold,
            "config": cfg,
            "graph_sha256": graph_sha,
            "best_epoch": best_epoch,
            "best_score": best_score,
            "ap_h5": best_aps[0],
            "ap_h10": best_aps[1],
            "ap_h15": best_aps[2],
            "best_sha256": file_sha256(best_path),
            "history": history,
            "train_episodes": int(
                len(set(episode_ids[train_idx]))
            ),
            "heldout_episodes": int(
                len(set(episode_ids[val_idx]))
            )
        }

        atomic_json(
            result,
            result_path
        )

        fold_results.append(result)

        if progress_path.exists():
            progress_path.unlink()

        print(
            f"{tag}: COMPLETE "
            f"| BEST EPOCH={best_epoch}"
            f" | SCORE={best_score:.5f}",
            flush=True
        )

    del nodes_scaled

# =============================================================================
# 8. RANK HYPERPARAMETER CONFIGURATIONS
# =============================================================================

summary_rows = []

for cfg in CONFIGS:

    matches = [
        r for r in fold_results
        if r["config"]["name"] == cfg["name"]
    ]

    if len(matches) != N_FOLDS:
        raise RuntimeError(
            f"Missing CV result: {cfg['name']}"
        )

    summary_rows.append({
        "config": cfg["name"],
        "mean_score": float(
            np.mean([
                r["best_score"] for r in matches
            ])
        ),
        "mean_H5_AP": float(
            np.mean([
                r["ap_h5"] for r in matches
            ])
        ),
        "mean_H10_AP": float(
            np.mean([
                r["ap_h10"] for r in matches
            ])
        ),
        "mean_H15_AP": float(
            np.mean([
                r["ap_h15"] for r in matches
            ])
        ),
        "median_best_epoch": int(
            np.median([
                r["best_epoch"] for r in matches
            ])
        )
    })

summary_rows.sort(
    key=lambda r: r["mean_score"],
    reverse=True
)

winner = summary_rows[0]["config"]

summary = {
    "graph_sha256": graph_sha,
    "validation_dataset_accessed": False,
    "test_dataset_accessed": False,
    "ranked_configurations": summary_rows,
    "winner": winner,
    "note": (
        "Fast 2-fold grouped CV. "
        "Fold-specific best-epoch scores are "
        "preliminary screening results."
    )
}

atomic_json(
    summary,
    SEARCH_DIR / "CV_SELECTION_SUMMARY.json"
)

# =============================================================================
# 9. FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 85)
print("TRAIN-ONLY GRAPH + LSTM OPTIMIZATION COMPLETE")
print("=" * 85)

print(
    pd.DataFrame(summary_rows)
    .round(5)
    .to_string(index=False)
)

print("\nSELECTED CONFIGURATION:", winner)

print(
    "\nSaved selection summary:",
    SEARCH_DIR / "CV_SELECTION_SUMMARY.json"
)

print("\nOriginal Graph + LSTM checkpoint modified: NO")
print("Official VALIDATION accessed: NO")
print("Official TEST accessed: NO")

print(
    "\nNEXT: Refit selected configuration "
    "once on full TRAIN."
)

print("=" * 85)



In [ ]:

# =============================================================================
# CELL 7 — FINAL FULL-TRAIN REFIT OF SELECTED GRAPH + LSTM
#
# Uses the winner from Cell 6: C02_wider_graph
# Fixed epochs from TRAIN-only CV selection
# Preserves original Graph + LSTM and all other models
# Resume-safe after completed epochs
#
# TRAIN ONLY — NO OFFICIAL VALIDATION OR TEST
# =============================================================================

from pathlib import Path
import json
import hashlib
import random
import os
import time

import numpy as np
import torch
import torch.nn as nn

from google.colab import drive

# =============================================================================
# 1. CONFIGURATION AND PATHS
# =============================================================================

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/network-attack-forecasting")

GRAPH_ROOT = ROOT / "Synthetic_NAF_v2_GraphCompanion"

TRAIN_DIR = GRAPH_ROOT / "TRAIN_only_v1"

GRAPH_FILE = TRAIN_DIR / "TRAIN_temporal_graphs.npz"
MANIFEST_FILE = TRAIN_DIR / "GRAPH_COMPANION_MANIFEST.json"

SEARCH_DIR = (
    GRAPH_ROOT / "models" / "Graph_LSTM" / "cv_search_v1"
)

SELECTION_FILE = SEARCH_DIR / "CV_SELECTION_SUMMARY.json"

# Separate destination for the optimized final checkpoint
FINAL_DIR = (
    GRAPH_ROOT / "models" / "Graph_LSTM" / "optimized_full_train_v1"
)

FINAL_FILE = FINAL_DIR / "GRAPH_LSTM_C02_OPTIMIZED_STATE.pt"
PROGRESS_FILE = FINAL_DIR / "GRAPH_LSTM_C02_PROGRESS.pt"
RECORD_FILE = FINAL_DIR / "GRAPH_LSTM_C02_TRAIN_RECORD.json"

SEED = 20261027
BATCH_SIZE = 256
WEIGHT_DECAY = 0.001

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.set_num_threads(min(4, os.cpu_count() or 1))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def atomic_save(obj, destination):
    temporary = destination.with_name(destination.name + ".tmp")
    torch.save(obj, temporary)
    temporary.replace(destination)


print("=" * 85)
print("CELL 7 — OPTIMIZED GRAPH + LSTM FULL-TRAIN REFIT")
print("=" * 85)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

# =============================================================================
# 2. VERIFY SEARCH RESULTS AND SELECT WINNER
# =============================================================================

for path in [GRAPH_FILE, MANIFEST_FILE, SELECTION_FILE]:
    if not path.is_file():
        raise FileNotFoundError(path)

manifest = json.loads(MANIFEST_FILE.read_text())
selection = json.loads(SELECTION_FILE.read_text())

graph_sha = sha256(GRAPH_FILE)

if graph_sha != manifest["artifact_sha256"]:
    raise RuntimeError("TRAIN graph dataset checksum mismatch.")

if selection["graph_sha256"] != graph_sha:
    raise RuntimeError("CV selection belongs to another graph dataset.")

if selection["winner"] != "C02_wider_graph":
    raise RuntimeError(
        "Unexpected CV winner. Inspect the selection summary."
    )

selected = next(
    row for row in selection["ranked_configurations"]
    if row["config"] == selection["winner"]
)

# Recover the exact configuration from the original CV protocol
protocol_file = SEARCH_DIR / "CV_SEARCH_PROTOCOL.json"

if not protocol_file.is_file():
    raise FileNotFoundError(protocol_file)

protocol = json.loads(protocol_file.read_text())

if protocol["dataset_sha256"] != graph_sha:
    raise RuntimeError("CV protocol dataset mismatch.")

config = next(
    cfg for cfg in protocol["configs"]
    if cfg["name"] == selection["winner"]
)

EPOCHS = int(selected["median_best_epoch"])

assert EPOCHS > 0
assert config["graph_hidden"] == 64
assert config["lstm_hidden"] == 64
assert config["dropout"] == 0.10
assert config["lr"] == 0.001

print("\nCV-selected configuration:", selection["winner"])
print("Graph hidden:", config["graph_hidden"])
print("LSTM hidden:", config["lstm_hidden"])
print("Dropout:", config["dropout"])
print("Learning rate:", config["lr"])
print("Fixed full-TRAIN epochs:", EPOCHS)
print("CV mean H5 AP:", selected["mean_H5_AP"])
print("CV mean selection score:", selected["mean_score"])

# =============================================================================
# 3. LOAD FULL TRAIN GRAPH DATA
# =============================================================================

with np.load(GRAPH_FILE, allow_pickle=False) as data:

    nodes_raw = data["node_frames"].astype(np.float32)
    edges_raw = data["edge_flow_frames"].astype(np.float32)
    end_frames = data["window_end_frame"].astype(np.int64)
    targets = data["y_horizons"].astype(np.float32)
    episodes = data["episode_id"].astype(str)

assert nodes_raw.shape[1:] == (8, 8)
assert edges_raw.shape == (len(nodes_raw), 8, 8)
assert targets.shape == (len(end_frames), 3)

assert len(end_frames) == len(episodes)
assert end_frames.min() >= 9
assert end_frames.max() < len(nodes_raw)

assert np.isfinite(nodes_raw).all()
assert np.isfinite(edges_raw).all()
assert (nodes_raw >= 0).all()
assert (edges_raw >= 0).all()

assert np.isin(targets, [0, 1]).all()
assert np.all(targets[:, 0] <= targets[:, 1])
assert np.all(targets[:, 1] <= targets[:, 2])

print("\nTRAIN data integrity: PASS")
print("Training windows:", len(end_frames))
print("Training episodes:", len(np.unique(episodes)))
print("Graph snapshots:", len(nodes_raw))

# =============================================================================
# 4. FIT FINAL PREPROCESSING ON FULL TRAIN ONLY
# =============================================================================

print("\nFitting full-TRAIN scaler...")

log_nodes = np.log1p(nodes_raw)

flat = log_nodes.reshape(-1, 8)

median = np.median(flat, axis=0).astype(np.float32)

q25, q75 = np.percentile(
    flat,
    [25, 75],
    axis=0
)

scale = np.maximum(
    q75 - q25,
    0.05
).astype(np.float32)

scaled_nodes = np.clip(
    (log_nodes - median) / scale,
    -8.0,
    8.0
).astype(np.float32)

edge_total = edges_raw.sum(
    axis=(1, 2),
    keepdims=True
)

scaled_edges = (
    edges_raw / np.maximum(edge_total, 1e-8)
).astype(np.float32)

del nodes_raw, edges_raw, log_nodes, flat, edge_total

assert np.isfinite(scaled_nodes).all()
assert np.isfinite(scaled_edges).all()

print("Node scaler: LOG1P + MEDIAN/IQR + CLIP8")
print("Edges: PER-GRAPH FLOW FRACTIONS")
print("Scaler fitted on TRAIN only: YES")

# =============================================================================
# 5. EXACT SAME GRAPH + LSTM MODEL ARCHITECTURE AS CELL 6
# =============================================================================

class WeightedGraphConv(nn.Module):

    def __init__(self, in_dim, out_dim):
        super().__init__()

        self.self_linear = nn.Linear(in_dim, out_dim)
        self.neighbor_linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, adjacency):

        incoming = torch.bmm(adjacency, x)

        return torch.relu(
            self.self_linear(x)
            + self.neighbor_linear(incoming)
        )


class GraphLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_dim=8,
        graph_hidden=64,
        lstm_hidden=64,
        dropout=0.1
    ):

        super().__init__()

        self.graph_conv1 = WeightedGraphConv(
            input_dim, graph_hidden
        )

        self.graph_conv2 = WeightedGraphConv(
            graph_hidden, graph_hidden
        )

        self.graph_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=graph_hidden * 2,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    def forward(self, nodes, edges):

        b, t, n, f = nodes.shape

        x = nodes.reshape(b * t, n, f)

        adjacency = edges.reshape(b * t, n, n)

        # Edges are stored as source -> target.
        # Incoming messages require target -> source.
        adjacency = adjacency.transpose(1, 2)

        degree = adjacency.sum(dim=-1, keepdim=True)

        adjacency = adjacency / degree.clamp(min=1e-8)

        h = self.graph_conv1(x, adjacency)
        h = self.graph_dropout(h)
        h = self.graph_conv2(h, adjacency)

        pooled = torch.cat(
            [
                h.mean(dim=1),
                h.max(dim=1).values
            ],
            dim=-1
        )

        sequence = pooled.reshape(b, t, -1)

        temporal, _ = self.lstm(sequence)

        logits = self.head(temporal[:, -1, :])

        hazards = torch.sigmoid(logits)

        probabilities = 1.0 - torch.cumprod(
            1.0 - hazards,
            dim=1
        )

        return probabilities.clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )

# =============================================================================
# 6. INITIALIZE OPTIMIZED MODEL
# =============================================================================

model = GraphLSTMForecaster(
    input_dim=8,
    graph_hidden=config["graph_hidden"],
    lstm_hidden=config["lstm_hidden"],
    dropout=config["dropout"]
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["lr"],
    weight_decay=WEIGHT_DECAY
)

criterion = nn.BCELoss()

parameter_count = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("\nOptimized model parameters:", parameter_count)

model.eval()

with torch.inference_mode():

    sample_nodes = torch.zeros(
        2, 10, 8, 8,
        device=DEVICE
    )

    sample_edges = torch.zeros(
        2, 10, 8, 8,
        device=DEVICE
    )

    test_output = model(sample_nodes, sample_edges)

    assert test_output.shape == (2, 3)
    assert torch.isfinite(test_output).all()
    assert torch.all(test_output[:, 0] <= test_output[:, 1])
    assert torch.all(test_output[:, 1] <= test_output[:, 2])

print("Forward pass and monotonicity: PASS")

# =============================================================================
# 7. RECOVER OR START TRAINING
# =============================================================================

FINAL_DIR.mkdir(parents=True, exist_ok=True)

start_epoch = 0
loss_history = []

if FINAL_FILE.exists():

    saved = torch.load(
        FINAL_FILE,
        map_location="cpu",
        weights_only=False
    )

    if saved["graph_sha256"] != graph_sha:
        raise RuntimeError("Existing final checkpoint dataset mismatch.")

    if saved["config"] != config:
        raise RuntimeError("Existing final checkpoint config mismatch.")

    if saved["epochs"] != EPOCHS:
        raise RuntimeError("Existing final checkpoint epoch mismatch.")

    print("\nOPTIMIZED GRAPH + LSTM ALREADY COMPLETE")
    print("Checkpoint:", FINAL_FILE)
    print("SHA256:", sha256(FINAL_FILE))
    print("No training or overwrite performed.")

else:

    if PROGRESS_FILE.exists():

        saved = torch.load(
            PROGRESS_FILE,
            map_location="cpu",
            weights_only=False
        )

        if saved["graph_sha256"] != graph_sha:
            raise RuntimeError("Progress dataset mismatch.")

        if saved["config"] != config:
            raise RuntimeError("Progress config mismatch.")

        if saved["epochs"] != EPOCHS:
            raise RuntimeError("Progress epoch mismatch.")

        if not np.array_equal(saved["median"], median):
            raise RuntimeError("Saved scaler median mismatch.")

        if not np.array_equal(saved["scale"], scale):
            raise RuntimeError("Saved scaler scale mismatch.")

        model.load_state_dict(saved["model_state"])
        optimizer.load_state_dict(saved["optimizer_state"])

        start_epoch = int(saved["completed_epoch"])
        loss_history = list(saved["loss_history"])

        # Restore dropout randomness for reproducible continuation.
        torch.set_rng_state(saved["torch_rng_state"])

        if (
            DEVICE.type == "cuda"
            and "cuda_rng_state" in saved
        ):
            torch.cuda.set_rng_state_all(
                saved["cuda_rng_state"]
            )

        print("\nResuming after epoch:", start_epoch)

    # =========================================================================
    # 8. TRAIN ON ALL FROZEN TRAIN WINDOWS
    # =========================================================================

    offsets = np.arange(-9, 1, dtype=np.int64)

    n = len(end_frames)

    print("\n" + "=" * 85)
    print("STARTING OPTIMIZED GRAPH + LSTM TRAINING")
    print("=" * 85)

    print("Epochs:", EPOCHS)
    print("Batch size:", BATCH_SIZE)
    print("Windows:", n)
    print("Device:", DEVICE)

    for epoch in range(start_epoch, EPOCHS):

        started = time.monotonic()

        model.train()

        rng = np.random.default_rng(SEED + epoch)

        order = rng.permutation(n)

        total_loss = 0.0
        total_seen = 0

        for start in range(0, n, BATCH_SIZE):

            ids = order[start:start + BATCH_SIZE]

            frame_idx = (
                end_frames[ids, None]
                + offsets[None, :]
            )

            xb_nodes = torch.from_numpy(
                scaled_nodes[frame_idx]
            ).to(DEVICE)

            xb_edges = torch.from_numpy(
                scaled_edges[frame_idx]
            ).to(DEVICE)

            yb = torch.from_numpy(
                targets[ids]
            ).to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            predictions = model(xb_nodes, xb_edges)

            loss = criterion(predictions, yb)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss at epoch {epoch + 1}"
                )

            loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_loss += loss.item() * len(ids)
            total_seen += len(ids)

        mean_loss = total_loss / total_seen

        loss_history.append(float(mean_loss))

        elapsed = time.monotonic() - started

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS}"
            f" | TRAIN BCE={mean_loss:.6f}"
            f" | seconds={elapsed:.1f}",
            flush=True
        )

        # Save progress after each completed epoch.
        progress = {
            "completed_epoch": epoch + 1,
            "epochs": EPOCHS,
            "config": config,
            "graph_sha256": graph_sha,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "median": median,
            "scale": scale,
            "loss_history": loss_history,
            "torch_rng_state": torch.get_rng_state()
        }

        if DEVICE.type == "cuda":
            progress["cuda_rng_state"] = (
                torch.cuda.get_rng_state_all()
            )

        atomic_save(progress, PROGRESS_FILE)

    # =========================================================================
    # 9. SAVE OPTIMIZED FINAL CHECKPOINT
    # =========================================================================

    model.eval()

    final_state = {
        "model_state": model.state_dict(),
        "median": median,
        "scale": scale,
        "graph_sha256": graph_sha,
        "config": config,
        "epochs": EPOCHS,
        "loss_history": loss_history,
        "architecture": "WeightedGraphConv_GraphConv_LSTM",
        "selection": "TRAIN_ONLY_EPISODE_GROUPED_CV",
        "graph_origin": "SYNTHETIC_FROM_AGGREGATE_V2",
        "validation_accessed": False,
        "test_accessed": False
    }

    atomic_save(final_state, FINAL_FILE)

    checkpoint_sha = sha256(FINAL_FILE)

    record = {
        "model": "GraphLSTMForecaster",
        "selected_config": selection["winner"],
        "config": config,
        "epochs": EPOCHS,
        "train_windows": n,
        "train_episodes": int(len(np.unique(episodes))),
        "parameters": parameter_count,
        "graph_sha256": graph_sha,
        "checkpoint_sha256": checkpoint_sha,
        "cv_mean_score": selected["mean_score"],
        "cv_mean_h5_ap": selected["mean_H5_AP"],
        "cv_mean_h10_ap": selected["mean_H10_AP"],
        "cv_mean_h15_ap": selected["mean_H15_AP"],
        "train_loss_history": loss_history,
        "final_train_bce": loss_history[-1],
        "validation_accessed": False,
        "test_accessed": False
    }

    record_temp = RECORD_FILE.with_name(
        RECORD_FILE.name + ".tmp"
    )

    record_temp.write_text(
        json.dumps(record, indent=2)
    )

    record_temp.replace(RECORD_FILE)

    print("\n" + "=" * 85)
    print("OPTIMIZED GRAPH + LSTM — FULL TRAIN COMPLETE")
    print("=" * 85)

    print("Selected configuration:", selection["winner"])
    print("Checkpoint:", FINAL_FILE)
    print("SHA256:", checkpoint_sha)
    print("Parameters:", parameter_count)
    print("Epochs:", EPOCHS)
    print("Final TRAIN BCE:", loss_history[-1])

    print("\nOriginal model overwritten: NO")
    print("Official VALIDATION accessed: NO")
    print("Official TEST accessed: NO")

    print("=" * 85)



In [ ]:

# =============================================================================
# CELL 8 — ORIGINAL vs OPTIMIZED GRAPH + LSTM
#
# Official VALIDATION comparison, 18,839 windows / 300 episodes
#
# Both models:
#   - Load their original saved checkpoints
#   - Reuse their own TRAIN-fitted preprocessing
#   - Predict identical validation windows
#
# Metrics:
#   AP, AUROC, Brier, Precision, Recall, F1, FPR
#   Episode-level false alerts
#   Episode-bootstrap confidence intervals for AP differences
#
# NO TRAINING
# NO SCALER FITTING
# NO TEST ACCESS
# =============================================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

ROOT = Path("/content/drive/MyDrive/network-attack-forecasting")
BASE = ROOT / "Synthetic_NAF_v2_GraphCompanion"

TRAIN_MANIFEST = (
    BASE / "TRAIN_only_v1" / "GRAPH_COMPANION_MANIFEST.json"
)

VAL_MANIFEST = (
    BASE / "VALIDATION_only_v1" /
    "GRAPH_COMPANION_VALIDATION_MANIFEST.json"
)

VAL_FILE = (
    BASE / "VALIDATION_only_v1" /
    "VALIDATION_temporal_graphs.npz"
)

ORIGINAL_FILE = (
    BASE / "models" / "Graph_LSTM" /
    "train_only_v1" / "GRAPH_LSTM_STATE.pt"
)

OPTIMIZED_FILE = (
    BASE / "models" / "Graph_LSTM" /
    "optimized_full_train_v1" /
    "GRAPH_LSTM_C02_OPTIMIZED_STATE.pt"
)

REPORT_DIR = (
    BASE / "models" / "Graph_LSTM" /
    "validation_comparison_v1"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 256
THRESHOLD = 0.5
BOOTSTRAP_REPEATS = 300
HORIZONS = [5, 10, 15]


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 90)
print("CELL 8 — ORIGINAL vs OPTIMIZED GRAPH + LSTM")
print("=" * 90)
print("Device:", DEVICE)

# =============================================================================
# 1. VERIFY INPUTS
# =============================================================================

for file in [
    TRAIN_MANIFEST, VAL_MANIFEST, VAL_FILE,
    ORIGINAL_FILE, OPTIMIZED_FILE
]:
    if not file.is_file():
        raise FileNotFoundError(file)

train_meta = json.loads(TRAIN_MANIFEST.read_text())
val_meta = json.loads(VAL_MANIFEST.read_text())

train_sha = train_meta["artifact_sha256"]
val_sha = sha256(VAL_FILE)

assert val_sha == val_meta["artifact_sha256"]

assert (
    val_meta["source_train_graph_sha256"]
    == train_sha
)

assert val_meta["generator_seed"] == train_meta["seed"]

print("TRAIN / VALIDATION graph provenance: PASS")

# =============================================================================
# 2. LOAD OFFICIAL VALIDATION GRAPH DATA
# =============================================================================

with np.load(VAL_FILE, allow_pickle=False) as d:
    nodes = d["node_frames"].astype(np.float32)
    edges = d["edge_flow_frames"].astype(np.float32)
    end_frames = d["window_end_frame"].astype(np.int64)
    y_true = d["y_horizons"].astype(np.int8)
    episodes = d["episode_id"].astype(str)
    minutes = d["minute"].astype(np.int32)

assert nodes.shape[1:] == (8, 8)
assert edges.shape == (len(nodes), 8, 8)
assert y_true.shape == (len(end_frames), 3)

assert len(end_frames) == len(episodes) == len(minutes)

assert np.isfinite(nodes).all()
assert np.isfinite(edges).all()

assert end_frames.min() >= 9
assert end_frames.max() < len(nodes)

assert np.isin(y_true, [0, 1]).all()

assert np.all(y_true[:, 0] <= y_true[:, 1])
assert np.all(y_true[:, 1] <= y_true[:, 2])

# Verify original frozen row labels and ordering.
frozen_val = (
    ROOT / "Synthetic_NAF_v2" /
    "release_candidate_corrected" /
    "windows_W10_VALIDATION.npz"
)

with np.load(frozen_val, allow_pickle=False) as d:
    assert np.array_equal(
        y_true, d["y_horizons"]
    )
    assert np.array_equal(
        episodes, d["episode_id"].astype(str)
    )
    assert np.array_equal(
        minutes, d["minute"]
    )

print("Frozen VALIDATION row alignment: PASS")
print("Validation windows:", len(end_frames))
print("Validation episodes:", len(np.unique(episodes)))
print("Temporal graph frames:", len(nodes))

# =============================================================================
# 3. RECREATE EXACT GRAPH + LSTM ARCHITECTURE
# =============================================================================

class WeightedGraphConv(nn.Module):

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.self_linear = nn.Linear(in_dim, out_dim)
        self.neighbor_linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, adjacency):
        incoming = torch.bmm(adjacency, x)
        return torch.relu(
            self.self_linear(x) +
            self.neighbor_linear(incoming)
        )


class GraphLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_dim=8,
        graph_hidden=32,
        lstm_hidden=64,
        dropout=0.1
    ):
        super().__init__()

        self.graph_conv1 = WeightedGraphConv(
            input_dim, graph_hidden
        )

        self.graph_conv2 = WeightedGraphConv(
            graph_hidden, graph_hidden
        )

        self.graph_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=2 * graph_hidden,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    def forward(self, node_data, edge_data):

        b, t, n, f = node_data.shape

        x = node_data.reshape(b * t, n, f)

        # Edge data: source -> target
        # Incoming messages: target -> source
        a = edge_data.reshape(b * t, n, n)
        a = a.transpose(1, 2)

        degree = a.sum(dim=-1, keepdim=True)
        a = a / degree.clamp(min=1e-8)

        h = self.graph_conv1(x, a)
        h = self.graph_dropout(h)
        h = self.graph_conv2(h, a)

        pooled = torch.cat(
            [h.mean(1), h.max(1).values],
            dim=-1
        )

        sequence = pooled.reshape(b, t, -1)

        temporal, _ = self.lstm(sequence)

        hazards = torch.sigmoid(
            self.head(temporal[:, -1, :])
        )

        return (
            1 - torch.cumprod(1 - hazards, dim=1)
        ).clamp(1e-6, 1 - 1e-6)


# =============================================================================
# 4. LOAD BOTH ORIGINAL CHECKPOINTS
# =============================================================================

def load_model(file):

    # These checkpoints were created by your own Colab code.
    saved = torch.load(
        file,
        map_location="cpu",
        weights_only=False
    )

    assert saved["graph_sha256"] == train_sha

    cfg = saved["config"]

    model = GraphLSTMForecaster(
        input_dim=8,
        graph_hidden=int(cfg["graph_hidden"]),
        lstm_hidden=int(cfg["lstm_hidden"]),
        dropout=float(cfg["dropout"])
    ).to(DEVICE)

    model.load_state_dict(
        saved["model_state"],
        strict=True
    )

    model.eval()

    median = np.asarray(
        saved["median"],
        dtype=np.float32
    )

    scale = np.asarray(
        saved["scale"],
        dtype=np.float32
    )

    assert median.shape == (8,)
    assert scale.shape == (8,)
    assert (scale > 0).all()

    return model, median, scale, cfg


original, original_median, original_scale, original_cfg = (
    load_model(ORIGINAL_FILE)
)

optimized, optimized_median, optimized_scale, optimized_cfg = (
    load_model(OPTIMIZED_FILE)
)

assert original_cfg["graph_hidden"] == 32
assert optimized_cfg["graph_hidden"] == 64

print("\nOriginal checkpoint: PASS")
print("Optimized C02 checkpoint: PASS")

print("Original SHA256:", sha256(ORIGINAL_FILE))
print("Optimized SHA256:", sha256(OPTIMIZED_FILE))

# =============================================================================
# 5. APPLY TRAIN-FITTED FEATURE TRANSFORMS
# =============================================================================

def preprocess_nodes(raw, median, scale):

    z = np.log1p(raw)

    z = np.clip(
        (z - median) / scale,
        -8.0,
        8.0
    )

    return z.astype(np.float32)


original_nodes = preprocess_nodes(
    nodes,
    original_median,
    original_scale
)

optimized_nodes = preprocess_nodes(
    nodes,
    optimized_median,
    optimized_scale
)

# Identical edge processing to the training code.
edge_total = edges.sum(
    axis=(1, 2),
    keepdims=True
)

edge_values = (
    edges / np.maximum(edge_total, 1e-8)
).astype(np.float32)

assert np.isfinite(original_nodes).all()
assert np.isfinite(optimized_nodes).all()
assert np.isfinite(edge_values).all()

print("\nTRAIN-fitted preprocessing reused: PASS")
print("Validation scaler refitted: NO")

# =============================================================================
# 6. INFERENCE ON IDENTICAL VALIDATION WINDOWS
# =============================================================================

offsets = np.arange(-9, 1, dtype=np.int64)


def predict(model, node_values, edge_values, label):

    print(f"\nPredicting: {label}")

    model.eval()
    output = []

    with torch.inference_mode():

        for start in range(
            0, len(end_frames), BATCH_SIZE
        ):
            stop = min(
                start + BATCH_SIZE,
                len(end_frames)
            )

            ids = np.arange(start, stop)

            frame_ids = (
                end_frames[ids, None]
                + offsets[None, :]
            )

            xb = torch.from_numpy(
                node_values[frame_ids]
            ).to(DEVICE)

            eb = torch.from_numpy(
                edge_values[frame_ids]
            ).to(DEVICE)

            p = model(xb, eb)

            output.append(p.cpu().numpy())

            if stop % 5000 < BATCH_SIZE or stop == len(end_frames):
                print(f"  {stop:,}/{len(end_frames):,}")

    probabilities = np.concatenate(
        output,
        axis=0
    ).astype(np.float64)

    assert probabilities.shape == y_true.shape
    assert np.isfinite(probabilities).all()
    assert (probabilities >= 0).all()
    assert (probabilities <= 1).all()

    assert np.all(
        probabilities[:, 0] <= probabilities[:, 1] + 1e-7
    )

    assert np.all(
        probabilities[:, 1] <= probabilities[:, 2] + 1e-7
    )

    return probabilities


p_original = predict(
    original,
    original_nodes,
    edge_values,
    "Original Graph + LSTM"
)

p_optimized = predict(
    optimized,
    optimized_nodes,
    edge_values,
    "Optimized C02 Graph + LSTM"
)

print("\nPrediction checks: PASS")

# =============================================================================
# 7. AP, AUROC, BRIER AND ALERT METRICS
# =============================================================================

rows = []


def calculate_metrics(name, probability):

    for j, h in enumerate(HORIZONS):

        y = y_true[:, j].astype(int)
        p = probability[:, j]

        yp = (p >= THRESHOLD).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y,
            yp,
            labels=[0, 1]
        ).ravel()

        rows.append({
            "Model": name,
            "Horizon": f"H{h}",
            "AP": average_precision_score(y, p),
            "AUROC": roc_auc_score(y, p),
            "Brier": brier_score_loss(y, p),
            "Precision_0.5": precision_score(
                y, yp, zero_division=0
            ),
            "Recall_0.5": recall_score(
                y, yp, zero_division=0
            ),
            "F1_0.5": f1_score(
                y, yp, zero_division=0
            ),
            "FPR_0.5": fp / max(fp + tn, 1),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn)
        })


calculate_metrics("Original", p_original)
calculate_metrics("Optimized C02", p_optimized)

metrics = pd.DataFrame(rows)

print("\n" + "=" * 90)
print("OFFICIAL VALIDATION — FULL METRICS")
print("=" * 90)

print(
    metrics.round(6).to_string(index=False)
)

# =============================================================================
# 8. DIRECT BEFORE/AFTER COMPARISON
# =============================================================================

comparison_rows = []

for j, h in enumerate(HORIZONS):

    y = y_true[:, j]

    before = average_precision_score(
        y, p_original[:, j]
    )

    after = average_precision_score(
        y, p_optimized[:, j]
    )

    comparison_rows.append({
        "Horizon": f"H{h}",
        "Original_AP": before,
        "Optimized_AP": after,
        "Delta_AP": after - before
    })

comparison = pd.DataFrame(comparison_rows)

print("\n" + "=" * 90)
print("OPTIMIZATION — AP CHANGE")
print("=" * 90)

print(
    comparison.round(6).to_string(index=False)
)

# =============================================================================
# 9. FALSE-ALERT RATE ON ENTIRELY NEGATIVE EPISODES
# =============================================================================

print("\n" + "=" * 90)
print("EPISODE-LEVEL FALSE ALERTS AT THRESHOLD 0.5")
print("=" * 90)

episode_alerts = []

for j, h in enumerate(HORIZONS):

    df = pd.DataFrame({
        "episode": episodes,
        "label": y_true[:, j],
        "original": p_original[:, j],
        "optimized": p_optimized[:, j]
    })

    grouped = df.groupby("episode").agg(
        label_max=("label", "max"),
        original_max=("original", "max"),
        optimized_max=("optimized", "max")
    )

    benign = grouped[
        grouped["label_max"] == 0
    ]

    if len(benign) == 0:
        continue

    orig_rate = float(
        (benign["original_max"] >= THRESHOLD).mean()
    )

    opt_rate = float(
        (benign["optimized_max"] >= THRESHOLD).mean()
    )

    episode_alerts.append({
        "Horizon": f"H{h}",
        "Negative_episodes": int(len(benign)),
        "Original_false_alert_rate": orig_rate,
        "Optimized_false_alert_rate": opt_rate
    })

    print(
        f"H{h}: Negative episodes={len(benign)} "
        f"| Original={orig_rate:.2%} "
        f"| Optimized={opt_rate:.2%}"
    )

# =============================================================================
# 10. PAIRED EPISODE-BOOTSTRAP AP DIFFERENCE
# =============================================================================

print("\n" + "=" * 90)
print("PAIRED EPISODE BOOTSTRAP — AP DIFFERENCE")
print("=" * 90)

# Resample entire episodes, never individual overlapping
# windows, to better respect within-episode dependence.
unique_ep, inverse = np.unique(
    episodes,
    return_inverse=True
)

episode_rows = [
    np.flatnonzero(inverse == k)
    for k in range(len(unique_ep))
]

rng = np.random.default_rng(20261027)

bootstrap_deltas = {
    h: [] for h in HORIZONS
}

for repeat in range(BOOTSTRAP_REPEATS):

    sampled_groups = rng.integers(
        0,
        len(unique_ep),
        size=len(unique_ep)
    )

    sampled_rows = np.concatenate(
        [episode_rows[k] for k in sampled_groups]
    )

    for j, h in enumerate(HORIZONS):

        ys = y_true[sampled_rows, j]

        if len(np.unique(ys)) != 2:
            continue

        ap_before = average_precision_score(
            ys,
            p_original[sampled_rows, j]
        )

        ap_after = average_precision_score(
            ys,
            p_optimized[sampled_rows, j]
        )

        bootstrap_deltas[h].append(
            ap_after - ap_before
        )

bootstrap_summary = []

for h in HORIZONS:

    differences = np.asarray(
        bootstrap_deltas[h]
    )

    if len(differences) == 0:
        raise RuntimeError(
            f"No valid bootstrap samples for H{h}"
        )

    lower, upper = np.percentile(
        differences,
        [2.5, 97.5]
    )

    actual_delta = comparison.loc[
        comparison["Horizon"] == f"H{h}",
        "Delta_AP"
    ].iloc[0]

    bootstrap_summary.append({
        "Horizon": f"H{h}",
        "Delta_AP": float(actual_delta),
        "CI95_lower": float(lower),
        "CI95_upper": float(upper),
        "Bootstrap_repeats": len(differences)
    })

bootstrap_table = pd.DataFrame(
    bootstrap_summary
)

print(
    bootstrap_table.round(6).to_string(index=False)
)

print(
    "\nInterpretation: A positive Delta_AP favors C02. "
    "If the interval crosses zero, the observed "
    "improvement is uncertain."
)

# =============================================================================
# 11. COMPARE AGAINST PREVIOUS BEST STACKED LSTM
# =============================================================================

stacked_reference = {
    "H5": 0.761005,
    "H10": 0.906018,
    "H15": 0.935854
}

print("\n" + "=" * 90)
print("COMPARISON WITH PREVIOUS STACKED LSTM REFERENCE")
print("=" * 90)

for j, h in enumerate(HORIZONS):

    optimized_ap = average_precision_score(
        y_true[:, j],
        p_optimized[:, j]
    )

    reference_ap = stacked_reference[f"H{h}"]

    print(
        f"H{h} | "
        f"Optimized Graph AP={optimized_ap:.6f} | "
        f"Stacked LSTM reference={reference_ap:.6f} | "
        f"Difference={optimized_ap - reference_ap:+.6f}"
    )

print(
    "\nStacked LSTM values are from earlier V2 "
    "validation results, not replayed here."
)

# =============================================================================
# 12. SAVE RESULTS SEPARATELY
# =============================================================================

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

prediction_file = (
    REPORT_DIR / "ORIGINAL_VS_C02_VAL_PREDICTIONS.npz"
)

metrics_file = (
    REPORT_DIR / "ORIGINAL_VS_C02_METRICS.csv"
)

report_file = (
    REPORT_DIR / "ORIGINAL_VS_C02_REPORT.json"
)

report = {
    "validation_graph_sha256": val_sha,
    "original_checkpoint_sha256": sha256(ORIGINAL_FILE),
    "optimized_checkpoint_sha256": sha256(OPTIMIZED_FILE),
    "windows": int(len(end_frames)),
    "episodes": int(len(unique_ep)),
    "threshold": THRESHOLD,
    "metrics": metrics.to_dict(orient="records"),
    "ap_comparison": comparison.to_dict(orient="records"),
    "episode_false_alerts": episode_alerts,
    "bootstrap": bootstrap_summary,
    "stacked_lstm_reference": stacked_reference,
    "graph_origin": "SYNTHETIC_FROM_AGGREGATE_V2",
    "validation_previously_examined": True,
    "test_accessed": False
}

# Never silently replace a prior evaluation artifact.
if any(
    p.exists()
    for p in [prediction_file, metrics_file, report_file]
):
    if not all(
        p.exists()
        for p in [prediction_file, metrics_file, report_file]
    ):
        raise RuntimeError(
            "Partial evaluation artifacts already exist. "
            "Inspect before rerunning."
        )

    previous = json.loads(report_file.read_text())

    for key in [
        "validation_graph_sha256",
        "original_checkpoint_sha256",
        "optimized_checkpoint_sha256"
    ]:
        if previous[key] != report[key]:
            raise RuntimeError(
                f"Existing evaluation uses different inputs: {key}"
            )

    print("\nMatching evaluation files already exist.")
    print("No artifacts overwritten.")

else:

    temp_prediction = (
        REPORT_DIR / "ORIGINAL_VS_C02_TEMP.npz"
    )

    np.savez_compressed(
        temp_prediction,
        y_horizons=y_true,
        episode_id=episodes,
        minute=minutes,
        original_probabilities=p_original,
        optimized_probabilities=p_optimized
    )

    temp_prediction.replace(prediction_file)

    metrics.to_csv(
        metrics_file,
        index=False
    )

    report_file.write_text(
        json.dumps(report, indent=2)
    )

    print("\nEvaluation artifacts saved.")

print("Predictions:", prediction_file)
print("Metrics:", metrics_file)
print("Report:", report_file)

print("\n" + "=" * 90)
print("CELL 8 — OPTIMIZED GRAPH VALIDATION COMPLETE")
print("=" * 90)

print("Original model evaluated: YES")
print("Optimized C02 evaluated: YES")
print("TRAIN checkpoints modified: NO")
print("Validation scaler refitted: NO")
print("Official TEST accessed: NO")
print("=" * 90)


In [ ]:

# =============================================================================
# CELL 9 — STANDARD GCN + LSTM FORECASTER
#
# Architecture:
# PyTorch Geometric GCNConv -> GCNConv
# -> Graph Pooling -> LSTM -> H5/H10/H15
#
# REAL GCNConv, not custom WeightedGraphConv.
#
# Reads TRAIN graph companion only.
# Saves new checkpoints separately.
# No official VALIDATION or TEST access.
# =============================================================================

from pathlib import Path
import sys
import subprocess
import hashlib
import json
import random
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive

# =============================================================================
# 1. INSTALL PYTORCH GEOMETRIC IF MISSING
# =============================================================================

try:
    import torch_geometric
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch_geometric"
    ])
    import torch_geometric

from torch_geometric.nn import GCNConv

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

BASE = ROOT / "Synthetic_NAF_v2_GraphCompanion"

TRAIN_FILE = (
    BASE / "TRAIN_only_v1" /
    "TRAIN_temporal_graphs.npz"
)

MANIFEST_FILE = (
    BASE / "TRAIN_only_v1" /
    "GRAPH_COMPANION_MANIFEST.json"
)

MODEL_DIR = (
    BASE / "models" /
    "GCN_LSTM" /
    "train_only_v1"
)

FINAL_FILE = MODEL_DIR / "GCN_LSTM_STATE.pt"
PROGRESS_FILE = MODEL_DIR / "GCN_LSTM_PROGRESS.pt"
RECORD_FILE = MODEL_DIR / "GCN_LSTM_TRAIN_RECORD.json"

SEED = 20261027

EPOCHS = 12
BATCH_SIZE = 128
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.001

GCN_HIDDEN = 32
LSTM_HIDDEN = 64
DROPOUT = 0.1

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
else:
    torch.set_num_threads(
        min(4, os.cpu_count() or 1)
    )


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def atomic_torch(obj, path):
    temp = path.with_name(path.name + ".tmp")
    torch.save(obj, temp)
    temp.replace(path)


CONFIG = {
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "gcn_hidden": GCN_HIDDEN,
    "lstm_hidden": LSTM_HIDDEN,
    "dropout": DROPOUT,
    "nodes": 8,
    "node_features": 8,
    "history": 10,
    "architecture": "PyG_GCNConv_GCNConv_LSTM",
    "edge_processing": "OUTGOING_ROW_NORMALIZED"
}

print("=" * 85)
print("CELL 9 — STANDARD GCN + LSTM TRAINING")
print("=" * 85)

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("Graph layer: torch_geometric.nn.GCNConv")

# =============================================================================
# 2. VERIFY TRAIN GRAPH DATA
# =============================================================================

for path in [TRAIN_FILE, MANIFEST_FILE]:
    if not path.is_file():
        raise FileNotFoundError(path)

manifest = json.loads(
    MANIFEST_FILE.read_text()
)

train_sha = sha256(TRAIN_FILE)

if train_sha != manifest["artifact_sha256"]:
    raise RuntimeError("TRAIN graph checksum mismatch.")

assert manifest["nodes_per_graph"] == 8
assert manifest["history_minutes"] == 10
assert manifest["forecast_horizons_minutes"] == [5, 10, 15]

with np.load(TRAIN_FILE, allow_pickle=False) as d:

    nodes = d["node_frames"].astype(np.float32)

    edges = d["edge_flow_frames"].astype(np.float32)

    end_frames = d["window_end_frame"].astype(np.int64)

    targets = d["y_horizons"].astype(np.float32)

    episodes = d["episode_id"].astype(str)

assert nodes.shape[1:] == (8, 8)
assert edges.shape == (len(nodes), 8, 8)
assert targets.shape == (len(end_frames), 3)

assert len(episodes) == len(end_frames)

assert end_frames.min() >= 9
assert end_frames.max() < len(nodes)

assert np.isfinite(nodes).all()
assert np.isfinite(edges).all()

assert (nodes >= 0).all()
assert (edges >= 0).all()

assert np.isin(targets, [0, 1]).all()

assert np.all(targets[:, 0] <= targets[:, 1])
assert np.all(targets[:, 1] <= targets[:, 2])

print("\nTRAIN graph integrity: PASS")
print("Training windows:", len(end_frames))
print("Episodes:", len(np.unique(episodes)))
print("Graph snapshots:", len(nodes))

# =============================================================================
# 3. FIT TRAIN-ONLY NODE SCALER
# =============================================================================

log_nodes = np.log1p(nodes)

flat = log_nodes.reshape(-1, 8)

median = np.median(
    flat,
    axis=0
).astype(np.float32)

q25, q75 = np.percentile(
    flat,
    [25, 75],
    axis=0
)

scale = np.maximum(
    q75 - q25,
    0.05
).astype(np.float32)

nodes_scaled = np.clip(
    (log_nodes - median) / scale,
    -8.0,
    8.0
).astype(np.float32)

del nodes, log_nodes, flat

# Normalize outgoing edge weights at every source node.
# This differs from the custom model's edge normalization,
# and is explicitly recorded for later VALIDATION inference.

outgoing = edges.sum(
    axis=-1,
    keepdims=True
)

edges_scaled = (
    edges /
    np.maximum(outgoing, 1e-8)
).astype(np.float32)

del edges, outgoing

assert np.isfinite(nodes_scaled).all()
assert np.isfinite(edges_scaled).all()

print("\nNode scaler fitted on TRAIN: YES")
print("Node scaling: LOG1P + MEDIAN/IQR + CLIP8")
print("Edges: OUTGOING ROW NORMALIZATION")

# =============================================================================
# 4. DEFINE STANDARD GCN + LSTM
# =============================================================================

class GCNLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_features=8,
        gcn_hidden=32,
        lstm_hidden=64,
        dropout=0.1
    ):
        super().__init__()

        # Official PyTorch Geometric GCN operators
        self.gcn1 = GCNConv(
            input_features,
            gcn_hidden,
            cached=False,
            normalize=True
        )

        self.gcn2 = GCNConv(
            gcn_hidden,
            gcn_hidden,
            cached=False,
            normalize=True
        )

        self.dropout = nn.Dropout(dropout)

        # Mean + max graph pooling
        self.lstm = nn.LSTM(
            input_size=2 * gcn_hidden,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    def forward(self, node_input, edge_input):

        # node_input: (B, 10, 8, 8)
        # edge_input: (B, 10, 8, 8)
        #
        # edge_input matrix convention:
        # [source_node, destination_node]

        b, t, n, f = node_input.shape

        graph_count = b * t

        # Flatten graph snapshots into one disconnected
        # PyG mini-batch, with unique node indices.

        x = node_input.reshape(
            graph_count * n,
            f
        )

        adjacency = edge_input.reshape(
            graph_count,
            n,
            n
        )

        graph_id, src, dst = torch.where(
            adjacency > 0
        )

        offsets = graph_id * n

        edge_index = torch.stack(
            [
                offsets + src,
                offsets + dst
            ],
            dim=0
        ).long()

        edge_weight = adjacency[
            graph_id,
            src,
            dst
        ]

        # PyG automatically supplies GCN self-loops.
        # Isolated nodes still retain their own features.

        h = self.gcn1(
            x,
            edge_index,
            edge_weight=edge_weight
        )

        h = F.relu(h)

        h = self.dropout(h)

        h = self.gcn2(
            h,
            edge_index,
            edge_weight=edge_weight
        )

        h = F.relu(h)

        # Recover graph node structure.
        h = h.reshape(
            graph_count,
            n,
            -1
        )

        # Graph-level representations for each minute.
        mean_pool = h.mean(dim=1)
        max_pool = h.max(dim=1).values

        graph_vectors = torch.cat(
            [mean_pool, max_pool],
            dim=-1
        )

        # Temporal sequence of graph representations.
        sequence = graph_vectors.reshape(
            b,
            t,
            -1
        )

        temporal_output, _ = self.lstm(
            sequence
        )

        final_state = temporal_output[:, -1]

        interval_hazards = torch.sigmoid(
            self.head(final_state)
        )

        # H5 <= H10 <= H15 by construction
        probabilities = (
            1.0
            - torch.cumprod(
                1.0 - interval_hazards,
                dim=1
            )
        )

        return probabilities.clamp(
            min=1e-6,
            max=1 - 1e-6
        )


model = GCNLSTMForecaster(
    input_features=8,
    gcn_hidden=GCN_HIDDEN,
    lstm_hidden=LSTM_HIDDEN,
    dropout=DROPOUT
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

criterion = nn.BCELoss()

parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nArchitecture:")
print(model)
print("Trainable parameters:", parameter_count)

# =============================================================================
# 5. GRAPH MESSAGE-PASSING SANITY CHECK
# =============================================================================

model.eval()

with torch.inference_mode():

    dummy_nodes = torch.rand(
        2, 10, 8, 8,
        device=DEVICE
    )

    # Directed ring graph
    dummy_edges = torch.zeros(
        2, 10, 8, 8,
        device=DEVICE
    )

    for src in range(8):
        dummy_edges[:, :, src, (src + 1) % 8] = 1.0

    dummy_predictions = model(
        dummy_nodes,
        dummy_edges
    )

    assert dummy_predictions.shape == (2, 3)

    assert torch.isfinite(dummy_predictions).all()

    assert torch.all(
        dummy_predictions[:, 0]
        <= dummy_predictions[:, 1]
    )

    assert torch.all(
        dummy_predictions[:, 1]
        <= dummy_predictions[:, 2]
    )

    # Verify PyG graph connectivity affects output.
    # Compares ring edges with isolated nodes.
    isolated = torch.zeros_like(dummy_edges)

    ring_output = model(
        dummy_nodes,
        dummy_edges
    )

    isolated_output = model(
        dummy_nodes,
        isolated
    )

    graph_difference = (
        ring_output - isolated_output
    ).abs().max().item()

print("\nGCN forward pass: PASS")
print("Horizon monotonicity: PASS")
print("Edge-sensitivity diagnostic:", graph_difference)

# =============================================================================
# 6. SAVE/RECOVER CHECKPOINTS
# =============================================================================

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

start_epoch = 0
loss_history = []

if FINAL_FILE.exists():

    saved = torch.load(
        FINAL_FILE,
        map_location="cpu",
        weights_only=False
    )

    if saved["graph_sha256"] != train_sha:
        raise RuntimeError("Final checkpoint dataset mismatch.")

    if saved["config"] != CONFIG:
        raise RuntimeError("Final checkpoint config mismatch.")

    print("\nGCN + LSTM already trained.")
    print("Final checkpoint:", FINAL_FILE)
    print("Checkpoint SHA256:", sha256(FINAL_FILE))
    print("No retraining performed.")

else:

    if PROGRESS_FILE.exists():

        progress = torch.load(
            PROGRESS_FILE,
            map_location=DEVICE,
            weights_only=False
        )

        if progress["graph_sha256"] != train_sha:
            raise RuntimeError(
                "Saved progress dataset mismatch."
            )

        if progress["config"] != CONFIG:
            raise RuntimeError(
                "Saved progress configuration mismatch."
            )

        if not np.array_equal(
            progress["median"],
            median
        ):
            raise RuntimeError("Scaler median mismatch.")

        if not np.array_equal(
            progress["scale"],
            scale
        ):
            raise RuntimeError("Scaler IQR mismatch.")

        model.load_state_dict(
            progress["model_state"]
        )

        optimizer.load_state_dict(
            progress["optimizer_state"]
        )

        start_epoch = int(
            progress["completed_epoch"]
        )

        loss_history = list(
            progress["loss_history"]
        )

        torch.set_rng_state(
            progress["torch_rng_state"].cpu()
        )

        if (
            DEVICE.type == "cuda"
            and "cuda_rng_state" in progress
        ):
            torch.cuda.set_rng_state_all(
                progress["cuda_rng_state"]
            )

        print("\nResuming from epoch:", start_epoch)

    # =========================================================================
    # 7. TRAIN GCN + LSTM
    # =========================================================================

    n = len(end_frames)

    offsets = np.arange(
        -9,
        1,
        dtype=np.int64
    )

    print("\n" + "=" * 85)
    print("STARTING STANDARD GCN + LSTM TRAINING")
    print("=" * 85)

    print("Training windows:", n)
    print("Epochs:", EPOCHS)
    print("Batch size:", BATCH_SIZE)
    print("Device:", DEVICE)

    for epoch in range(start_epoch, EPOCHS):

        started = time.monotonic()

        model.train()

        order = np.random.default_rng(
            SEED + epoch
        ).permutation(n)

        running_loss = 0.0
        seen = 0

        for start in range(
            0,
            n,
            BATCH_SIZE
        ):

            ids = order[
                start:start + BATCH_SIZE
            ]

            frame_ids = (
                end_frames[ids, None]
                + offsets[None, :]
            )

            xb = torch.from_numpy(
                nodes_scaled[frame_ids]
            ).to(DEVICE)

            eb = torch.from_numpy(
                edges_scaled[frame_ids]
            ).to(DEVICE)

            yb = torch.from_numpy(
                targets[ids]
            ).to(DEVICE)

            optimizer.zero_grad(
                set_to_none=True
            )

            predictions = model(
                xb,
                eb
            )

            loss = criterion(
                predictions,
                yb
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Nonfinite loss at epoch {epoch+1}"
                )

            loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            running_loss += (
                loss.item() * len(ids)
            )

            seen += len(ids)

        epoch_loss = running_loss / seen

        loss_history.append(
            float(epoch_loss)
        )

        elapsed = (
            time.monotonic() - started
        )

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS}"
            f" | TRAIN BCE={epoch_loss:.6f}"
            f" | seconds={elapsed:.1f}",
            flush=True
        )

        # Save progress after every completed epoch.
        progress = {
            "completed_epoch": epoch + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "median": median,
            "scale": scale,
            "graph_sha256": train_sha,
            "config": CONFIG,
            "loss_history": loss_history,
            "torch_rng_state": torch.get_rng_state()
        }

        if DEVICE.type == "cuda":
            progress["cuda_rng_state"] = (
                torch.cuda.get_rng_state_all()
            )

        atomic_torch(
            progress,
            PROGRESS_FILE
        )

    # =========================================================================
    # 8. SAVE FINAL MODEL
    # =========================================================================

    model.eval()

    final_checkpoint = {
        "model_state": model.state_dict(),
        "median": median,
        "scale": scale,
        "graph_sha256": train_sha,
        "config": CONFIG,
        "loss_history": loss_history,
        "architecture": "PyG_GCNConv_GCNConv_LSTM",
        "node_origin": "SYNTHETIC",
        "graph_origin": "SYNTHETIC",
        "data_split": "TRAIN_ONLY",
        "validation_accessed": False,
        "test_accessed": False,
        "torch_version": torch.__version__,
        "pyg_version": torch_geometric.__version__
    }

    atomic_torch(
        final_checkpoint,
        FINAL_FILE
    )

    record = {
        "model": "GCNLSTMForecaster",
        "architecture": "PyG_GCNConv_GCNConv_LSTM",
        "config": CONFIG,
        "parameters": parameter_count,
        "training_windows": n,
        "training_episodes": int(
            len(np.unique(episodes))
        ),
        "train_graph_sha256": train_sha,
        "checkpoint_sha256": sha256(FINAL_FILE),
        "training_bce": loss_history,
        "final_train_bce": loss_history[-1],
        "synthetic_topology": True,
        "official_validation_accessed": False,
        "official_test_accessed": False
    }

    temporary_record = RECORD_FILE.with_name(
        RECORD_FILE.name + ".tmp"
    )

    temporary_record.write_text(
        json.dumps(record, indent=2)
    )

    temporary_record.replace(
        RECORD_FILE
    )

    print("\n" + "=" * 85)
    print("GCN + LSTM — TRAINING COMPLETE")
    print("=" * 85)

    print("Checkpoint:", FINAL_FILE)
    print("SHA256:", sha256(FINAL_FILE))
    print("Trainable parameters:", parameter_count)
    print("Training windows:", n)
    print("Final TRAIN BCE:", loss_history[-1])

    print("\nOriginal graph checkpoints modified: NO")
    print("Official VALIDATION accessed: NO")
    print("Official TEST accessed: NO")

    print("=" * 85)



In [ ]:

# =============================================================================
# CELL 10 — STANDARD GCN + LSTM VALIDATION
#
# Evaluate GCN + LSTM on frozen V2 VALIDATION.
#
# Compare with:
#   1. Original custom Graph + LSTM (saved predictions)
#   2. Optimized C02 Graph + LSTM (saved predictions)
#   3. Stacked LSTM (previously reported AP reference)
#
# No training.
# No scaler refitting.
# No TEST access.
# =============================================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from torch_geometric.nn import GCNConv

# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

BASE = ROOT / "Synthetic_NAF_v2_GraphCompanion"

VAL_FILE = (
    BASE / "VALIDATION_only_v1" /
    "VALIDATION_temporal_graphs.npz"
)

VAL_MANIFEST = (
    BASE / "VALIDATION_only_v1" /
    "GRAPH_COMPANION_VALIDATION_MANIFEST.json"
)

TRAIN_MANIFEST = (
    BASE / "TRAIN_only_v1" /
    "GRAPH_COMPANION_MANIFEST.json"
)

GCN_FILE = (
    BASE / "models" /
    "GCN_LSTM" /
    "train_only_v1" /
    "GCN_LSTM_STATE.pt"
)

PREVIOUS_PREDICTIONS = (
    BASE / "models" /
    "Graph_LSTM" /
    "validation_comparison_v1" /
    "ORIGINAL_VS_C02_VAL_PREDICTIONS.npz"
)

PREVIOUS_REPORT = (
    BASE / "models" /
    "Graph_LSTM" /
    "validation_comparison_v1" /
    "ORIGINAL_VS_C02_REPORT.json"
)

OUTPUT_DIR = (
    BASE / "models" /
    "GCN_LSTM" /
    "validation_v1"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 128
THRESHOLD = 0.5
HORIZONS = [5, 10, 15]
BOOTSTRAP_REPEATS = 300


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()


print("=" * 90)
print("CELL 10 — STANDARD GCN + LSTM VALIDATION")
print("=" * 90)
print("Device:", DEVICE)

# =============================================================================
# 2. VERIFY INPUTS
# =============================================================================

required = [
    VAL_FILE,
    VAL_MANIFEST,
    TRAIN_MANIFEST,
    GCN_FILE,
    PREVIOUS_PREDICTIONS,
    PREVIOUS_REPORT
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)

train_meta = json.loads(
    TRAIN_MANIFEST.read_text()
)

val_meta = json.loads(
    VAL_MANIFEST.read_text()
)

prior_report = json.loads(
    PREVIOUS_REPORT.read_text()
)

val_sha = sha256(VAL_FILE)
train_sha = train_meta["artifact_sha256"]

assert val_sha == val_meta["artifact_sha256"]

assert (
    val_meta["source_train_graph_sha256"]
    == train_sha
)

assert val_meta["generator_seed"] == train_meta["seed"]

assert (
    prior_report["validation_graph_sha256"]
    == val_sha
)

print("\nGraph provenance: PASS")

# =============================================================================
# 3. LOAD VALIDATION GRAPHS
# =============================================================================

with np.load(VAL_FILE, allow_pickle=False) as d:

    nodes = d["node_frames"].astype(np.float32)

    edges = d["edge_flow_frames"].astype(np.float32)

    end_frames = d["window_end_frame"].astype(np.int64)

    y_true = d["y_horizons"].astype(np.int8)

    episodes = d["episode_id"].astype(str)

    minutes = d["minute"].astype(np.int32)

assert nodes.shape[1:] == (8, 8)
assert edges.shape == (len(nodes), 8, 8)

assert y_true.shape == (len(end_frames), 3)

assert len(end_frames) == len(episodes) == len(minutes)

assert end_frames.min() >= 9
assert end_frames.max() < len(nodes)

assert np.isfinite(nodes).all()
assert np.isfinite(edges).all()

assert np.isin(y_true, [0, 1]).all()

assert np.all(y_true[:, 0] <= y_true[:, 1])
assert np.all(y_true[:, 1] <= y_true[:, 2])

print("Validation windows:", len(end_frames))
print("Validation episodes:", len(np.unique(episodes)))
print("Graph snapshots:", len(nodes))

# =============================================================================
# 4. VERIFY PREVIOUS GRAPH MODEL PREDICTIONS
# =============================================================================

with np.load(
    PREVIOUS_PREDICTIONS,
    allow_pickle=False
) as d:

    prior_y = d["y_horizons"]
    prior_episodes = d["episode_id"].astype(str)
    prior_minutes = d["minute"]

    original_probs = d[
        "original_probabilities"
    ].astype(np.float64)

    optimized_probs = d[
        "optimized_probabilities"
    ].astype(np.float64)

assert np.array_equal(y_true, prior_y)
assert np.array_equal(episodes, prior_episodes)
assert np.array_equal(minutes, prior_minutes)

assert original_probs.shape == y_true.shape
assert optimized_probs.shape == y_true.shape

assert np.isfinite(original_probs).all()
assert np.isfinite(optimized_probs).all()

print("Previous prediction row alignment: PASS")

# =============================================================================
# 5. RESTORE EXACT TRAINED GCN ARCHITECTURE
# =============================================================================

class GCNLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_features=8,
        gcn_hidden=32,
        lstm_hidden=64,
        dropout=0.1
    ):
        super().__init__()

        self.gcn1 = GCNConv(
            input_features,
            gcn_hidden,
            cached=False,
            normalize=True
        )

        self.gcn2 = GCNConv(
            gcn_hidden,
            gcn_hidden,
            cached=False,
            normalize=True
        )

        self.dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=2 * gcn_hidden,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    def forward(self, node_input, edge_input):

        b, t, n, f = node_input.shape

        graph_count = b * t

        x = node_input.reshape(
            graph_count * n,
            f
        )

        adjacency = edge_input.reshape(
            graph_count,
            n,
            n
        )

        graph_id, src, dst = torch.where(
            adjacency > 0
        )

        offsets = graph_id * n

        edge_index = torch.stack(
            [
                offsets + src,
                offsets + dst
            ],
            dim=0
        ).long()

        edge_weight = adjacency[
            graph_id,
            src,
            dst
        ]

        h = self.gcn1(
            x,
            edge_index,
            edge_weight=edge_weight
        )

        h = F.relu(h)

        h = self.dropout(h)

        h = self.gcn2(
            h,
            edge_index,
            edge_weight=edge_weight
        )

        h = F.relu(h)

        h = h.reshape(
            graph_count,
            n,
            -1
        )

        graph_vectors = torch.cat(
            [
                h.mean(dim=1),
                h.max(dim=1).values
            ],
            dim=-1
        )

        sequence = graph_vectors.reshape(
            b,
            t,
            -1
        )

        temporal_output, _ = self.lstm(
            sequence
        )

        final_state = temporal_output[:, -1]

        hazards = torch.sigmoid(
            self.head(final_state)
        )

        return (
            1.0
            - torch.cumprod(
                1.0 - hazards,
                dim=1
            )
        ).clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )


# =============================================================================
# 6. LOAD SAVED GCN CHECKPOINT
# =============================================================================

# Load only your own trusted training checkpoint.
checkpoint = torch.load(
    GCN_FILE,
    map_location="cpu",
    weights_only=False
)

assert checkpoint["graph_sha256"] == train_sha

cfg = checkpoint["config"]

assert cfg["architecture"] == "PyG_GCNConv_GCNConv_LSTM"
assert cfg["edge_processing"] == "OUTGOING_ROW_NORMALIZED"
assert cfg["history"] == 10
assert cfg["nodes"] == 8

model = GCNLSTMForecaster(
    input_features=cfg["node_features"],
    gcn_hidden=cfg["gcn_hidden"],
    lstm_hidden=cfg["lstm_hidden"],
    dropout=cfg["dropout"]
).to(DEVICE)

model.load_state_dict(
    checkpoint["model_state"],
    strict=True
)

model.eval()

print("\nGCN checkpoint loaded: PASS")
print("GCN SHA256:", sha256(GCN_FILE))

# =============================================================================
# 7. APPLY TRAIN-FITTED NODE SCALER
# =============================================================================

median = np.asarray(
    checkpoint["median"],
    dtype=np.float32
)

scale = np.asarray(
    checkpoint["scale"],
    dtype=np.float32
)

assert median.shape == (8,)
assert scale.shape == (8,)

assert np.isfinite(median).all()
assert np.isfinite(scale).all()
assert (scale > 0).all()

nodes_scaled = np.clip(
    (np.log1p(nodes) - median) / scale,
    -8.0,
    8.0
).astype(np.float32)

# IDENTICAL to GCN training:
# normalize each source node's outgoing edges.

outgoing = edges.sum(
    axis=-1,
    keepdims=True
)

edges_scaled = (
    edges / np.maximum(outgoing, 1e-8)
).astype(np.float32)

assert np.isfinite(nodes_scaled).all()
assert np.isfinite(edges_scaled).all()

print("Original TRAIN-fitted node scaler: PASS")
print("Edge normalization matches training: PASS")
print("Validation scaler refitting: NO")

# =============================================================================
# 8. GCN INFERENCE
# =============================================================================

offsets = np.arange(-9, 1, dtype=np.int64)

predictions = []

print("\nRunning GCN inference...")

with torch.inference_mode():

    for start in range(
        0,
        len(end_frames),
        BATCH_SIZE
    ):

        stop = min(
            start + BATCH_SIZE,
            len(end_frames)
        )

        ids = np.arange(start, stop)

        frame_ids = (
            end_frames[ids, None]
            + offsets[None, :]
        )

        xb = torch.from_numpy(
            nodes_scaled[frame_ids]
        ).to(DEVICE)

        eb = torch.from_numpy(
            edges_scaled[frame_ids]
        ).to(DEVICE)

        probs = model(xb, eb)

        predictions.append(
            probs.cpu().numpy()
        )

        if stop % 5000 < BATCH_SIZE or stop == len(end_frames):
            print(
                f"  {stop:,}/{len(end_frames):,}"
            )

gcn_probs = np.concatenate(
    predictions,
    axis=0
).astype(np.float64)

assert gcn_probs.shape == y_true.shape

assert np.isfinite(gcn_probs).all()

assert (gcn_probs >= 0).all()
assert (gcn_probs <= 1).all()

assert np.all(
    gcn_probs[:, 0] <= gcn_probs[:, 1] + 1e-7
)

assert np.all(
    gcn_probs[:, 1] <= gcn_probs[:, 2] + 1e-7
)

print("\nGCN inference: PASS")
print("Horizon monotonicity: PASS")

# =============================================================================
# 9. EVALUATE ALL THREE GRAPH MODELS
# =============================================================================

model_predictions = {
    "Original Graph LSTM": original_probs,
    "Optimized Graph LSTM": optimized_probs,
    "GCN + LSTM": gcn_probs
}

metric_rows = []

for name, probabilities in model_predictions.items():

    for j, h in enumerate(HORIZONS):

        y = y_true[:, j].astype(int)
        p = probabilities[:, j]

        binary = (p >= THRESHOLD).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y,
            binary,
            labels=[0, 1]
        ).ravel()

        metric_rows.append({
            "Model": name,
            "Horizon": f"H{h}",
            "AP": average_precision_score(y, p),
            "AUROC": roc_auc_score(y, p),
            "Brier": brier_score_loss(y, p),
            "Precision@0.5": precision_score(
                y, binary, zero_division=0
            ),
            "Recall@0.5": recall_score(
                y, binary, zero_division=0
            ),
            "F1@0.5": f1_score(
                y, binary, zero_division=0
            ),
            "FPR@0.5": fp / max(fp + tn, 1),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn)
        })

metrics_df = pd.DataFrame(metric_rows)

print("\n" + "=" * 90)
print("OFFICIAL VALIDATION — GRAPH MODEL METRICS")
print("=" * 90)

print(
    metrics_df[
        [
            "Model",
            "Horizon",
            "AP",
            "AUROC",
            "Brier",
            "Precision@0.5",
            "Recall@0.5",
            "F1@0.5",
            "FPR@0.5"
        ]
    ].round(6).to_string(index=False)
)

# =============================================================================
# 10. EPISODE-LEVEL FALSE ALERTS
# =============================================================================

print("\n" + "=" * 90)
print("EPISODE-LEVEL FALSE ALERTS @ 0.5")
print("=" * 90)

false_alert_rows = []

for j, h in enumerate(HORIZONS):

    frame = pd.DataFrame({
        "episode": episodes,
        "label": y_true[:, j],
        "original": original_probs[:, j],
        "optimized": optimized_probs[:, j],
        "gcn": gcn_probs[:, j]
    })

    grouped = frame.groupby("episode").agg(
        label_max=("label", "max"),
        original_max=("original", "max"),
        optimized_max=("optimized", "max"),
        gcn_max=("gcn", "max")
    )

    negative = grouped[
        grouped["label_max"] == 0
    ]

    for name, col in [
        ("Original Graph LSTM", "original_max"),
        ("Optimized Graph LSTM", "optimized_max"),
        ("GCN + LSTM", "gcn_max")
    ]:

        rate = (
            float(
                (negative[col] >= THRESHOLD).mean()
            )
            if len(negative) else np.nan
        )

        false_alert_rows.append({
            "Model": name,
            "Horizon": f"H{h}",
            "Negative_episodes": len(negative),
            "False_alert_rate": rate
        })

false_alert_df = pd.DataFrame(
    false_alert_rows
)

print(
    false_alert_df.round(6).to_string(index=False)
)

# =============================================================================
# 11. PAIRED EPISODE-BOOTSTRAP GCN vs ORIGINAL GRAPH LSTM
# =============================================================================

print("\n" + "=" * 90)
print("GCN vs ORIGINAL GRAPH LSTM — AP DIFFERENCES")
print("=" * 90)

unique_ep, inverse = np.unique(
    episodes,
    return_inverse=True
)

rows_by_episode = [
    np.flatnonzero(inverse == k)
    for k in range(len(unique_ep))
]

rng = np.random.default_rng(20261027)

bootstrap_rows = []

for j, h in enumerate(HORIZONS):

    y = y_true[:, j]

    original_ap = average_precision_score(
        y,
        original_probs[:, j]
    )

    gcn_ap = average_precision_score(
        y,
        gcn_probs[:, j]
    )

    delta = gcn_ap - original_ap

    bootstrap_deltas = []

    for _ in range(BOOTSTRAP_REPEATS):

        sampled_ep = rng.integers(
            0,
            len(unique_ep),
            size=len(unique_ep)
        )

        sampled_rows = np.concatenate(
            [rows_by_episode[k] for k in sampled_ep]
        )

        ys = y_true[sampled_rows, j]

        if np.unique(ys).size != 2:
            continue

        before = average_precision_score(
            ys,
            original_probs[sampled_rows, j]
        )

        after = average_precision_score(
            ys,
            gcn_probs[sampled_rows, j]
        )

        bootstrap_deltas.append(after - before)

    if len(bootstrap_deltas) == 0:
        raise RuntimeError(
            f"No valid bootstrap repetitions for H{h}"
        )

    lo, hi = np.percentile(
        bootstrap_deltas,
        [2.5, 97.5]
    )

    bootstrap_rows.append({
        "Horizon": f"H{h}",
        "Original_AP": original_ap,
        "GCN_AP": gcn_ap,
        "Delta_AP": delta,
        "CI95_lower": float(lo),
        "CI95_upper": float(hi)
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)

print(
    bootstrap_df.round(6).to_string(index=False)
)

# =============================================================================
# 12. STACKED LSTM REFERENCE
# =============================================================================

stacked_reference = {
    "H5": 0.761005,
    "H10": 0.906018,
    "H15": 0.935854
}

print("\n" + "=" * 90)
print("GCN vs STACKED LSTM — REPORTED VALIDATION AP")
print("=" * 90)

for j, h in enumerate(HORIZONS):

    gcn_ap = average_precision_score(
        y_true[:, j],
        gcn_probs[:, j]
    )

    baseline = stacked_reference[f"H{h}"]

    print(
        f"H{h}: "
        f"GCN={gcn_ap:.6f} "
        f"| Stacked={baseline:.6f} "
        f"| Delta={gcn_ap - baseline:+.6f}"
    )

print(
    "\nStacked LSTM scores are historical references, "
    "not newly replayed predictions."
)

# =============================================================================
# 13. SAVE THE GCN VALIDATION REPORT
# =============================================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PRED_FILE = OUTPUT_DIR / "GCN_VAL_PREDICTIONS.npz"
METRIC_FILE = OUTPUT_DIR / "GCN_VAL_METRICS.csv"
REPORT_FILE = OUTPUT_DIR / "GCN_VAL_REPORT.json"

report = {
    "model": "PyG_GCNConv_GCNConv_LSTM",
    "gcn_checkpoint_sha256": sha256(GCN_FILE),
    "validation_graph_sha256": val_sha,
    "windows": int(len(y_true)),
    "episodes": int(len(unique_ep)),
    "threshold": THRESHOLD,
    "metrics": metric_rows,
    "false_alerts": false_alert_rows,
    "bootstrap": bootstrap_rows,
    "stacked_lstm_reference": stacked_reference,
    "graph_origin": "SYNTHETIC",
    "validation_previously_examined": True,
    "test_accessed": False
}

if any(
    p.exists()
    for p in [PRED_FILE, METRIC_FILE, REPORT_FILE]
):

    if not all(
        p.exists()
        for p in [PRED_FILE, METRIC_FILE, REPORT_FILE]
    ):
        raise RuntimeError(
            "Partial previous GCN report exists. Inspect before rerun."
        )

    old = json.loads(REPORT_FILE.read_text())

    assert (
        old["gcn_checkpoint_sha256"]
        == report["gcn_checkpoint_sha256"]
    )

    assert (
        old["validation_graph_sha256"]
        == report["validation_graph_sha256"]
    )

    print("\nMatching GCN evaluation report already exists.")
    print("No artifacts overwritten.")

else:

    temporary = OUTPUT_DIR / "GCN_VAL_PREDICTIONS.tmp.npz"

    np.savez_compressed(
        temporary,
        y_horizons=y_true,
        episode_id=episodes,
        minute=minutes,
        gcn_probabilities=gcn_probs,
        original_graph_probabilities=original_probs,
        optimized_graph_probabilities=optimized_probs
    )

    temporary.replace(PRED_FILE)

    metrics_df.to_csv(
        METRIC_FILE,
        index=False
    )

    REPORT_FILE.write_text(
        json.dumps(report, indent=2)
    )

    print("\nEvaluation artifacts saved.")

print("\nPredictions:", PRED_FILE)
print("Metrics:", METRIC_FILE)
print("Report:", REPORT_FILE)

print("\n" + "=" * 90)
print("GCN + LSTM VALIDATION: COMPLETE")
print("=" * 90)

print("GCN model evaluated: YES")
print("Original graph model compared: YES")
print("Optimized graph model compared: YES")
print("Frozen TRAIN checkpoint modified: NO")
print("Scaler refitted: NO")
print("Official TEST accessed: NO")
print("=" * 90)



In [ ]:
# =============================================================================
# CELL 11 — GAT + LSTM TRAINING
#
# Architecture:
# GATConv -> GATConv -> Graph Pooling -> LSTM -> H5/H10/H15
#
# Uses PyTorch Geometric GATConv with edge attributes.
# Uses synthetic network graph companion.
#
# TRAIN ONLY
# NO official VALIDATION access
# NO official TEST access
# Existing checkpoints remain untouched
# =============================================================================

from pathlib import Path
import sys
import subprocess
import hashlib
import json
import random
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive

# =============================================================================
# 1. ENVIRONMENT
# =============================================================================

try:
    import torch_geometric
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "install", "-q", "torch_geometric"
    ])
    import torch_geometric

from torch_geometric.nn import GATConv

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

BASE = ROOT / "Synthetic_NAF_v2_GraphCompanion"

TRAIN_FILE = (
    BASE / "TRAIN_only_v1" /
    "TRAIN_temporal_graphs.npz"
)

MANIFEST_FILE = (
    BASE / "TRAIN_only_v1" /
    "GRAPH_COMPANION_MANIFEST.json"
)

MODEL_DIR = (
    BASE / "models" /
    "GAT_LSTM" /
    "train_only_v1"
)

FINAL_FILE = MODEL_DIR / "GAT_LSTM_STATE.pt"
PROGRESS_FILE = MODEL_DIR / "GAT_LSTM_PROGRESS.pt"
RECORD_FILE = MODEL_DIR / "GAT_LSTM_TRAIN_RECORD.json"

SEED = 20261027

EPOCHS = 12
BATCH_SIZE = 128

LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.001

GAT_HIDDEN = 32
ATTENTION_HEADS = 2
LSTM_HIDDEN = 64
DROPOUT = 0.1

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
else:
    torch.set_num_threads(
        min(4, os.cpu_count() or 1)
    )


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()


def atomic_save(obj, path):
    temp = path.with_name(path.name + ".tmp")
    torch.save(obj, temp)
    temp.replace(path)


CONFIG = {
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "gat_hidden": GAT_HIDDEN,
    "attention_heads": ATTENTION_HEADS,
    "lstm_hidden": LSTM_HIDDEN,
    "dropout": DROPOUT,
    "nodes": 8,
    "node_features": 8,
    "history": 10,
    "architecture": "PyG_GATConv_GATConv_LSTM",
    "edge_processing": "OUTGOING_ROW_NORMALIZED",
    "edge_attributes": "NORMALIZED_EDGE_FLOW",
    "gat_concat": False,
    "gat_self_loop_fill": 0.0
}

print("=" * 85)
print("CELL 11 — STANDARD GAT + LSTM TRAINING")
print("=" * 85)

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("Graph operator: GATConv")
print("Attention heads:", ATTENTION_HEADS)

# =============================================================================
# 2. VERIFY TRAIN INPUTS
# =============================================================================

for path in [TRAIN_FILE, MANIFEST_FILE]:
    if not path.is_file():
        raise FileNotFoundError(path)

manifest = json.loads(
    MANIFEST_FILE.read_text()
)

train_sha = sha256(TRAIN_FILE)

if train_sha != manifest["artifact_sha256"]:
    raise RuntimeError(
        "Graph TRAIN checksum mismatch."
    )

assert manifest["nodes_per_graph"] == 8
assert manifest["history_minutes"] == 10
assert manifest["forecast_horizons_minutes"] == [5, 10, 15]

with np.load(TRAIN_FILE, allow_pickle=False) as d:

    nodes_raw = d["node_frames"].astype(np.float32)

    edges_raw = d["edge_flow_frames"].astype(np.float32)

    end_frames = d["window_end_frame"].astype(np.int64)

    targets = d["y_horizons"].astype(np.float32)

    episodes = d["episode_id"].astype(str)

assert nodes_raw.shape[1:] == (8, 8)

assert edges_raw.shape == (
    len(nodes_raw), 8, 8
)

assert targets.shape == (len(end_frames), 3)

assert len(episodes) == len(end_frames)

assert end_frames.min() >= 9
assert end_frames.max() < len(nodes_raw)

assert np.isfinite(nodes_raw).all()
assert np.isfinite(edges_raw).all()

assert (nodes_raw >= 0).all()
assert (edges_raw >= 0).all()

assert np.isin(targets, [0, 1]).all()

assert np.all(
    targets[:, 0] <= targets[:, 1]
)

assert np.all(
    targets[:, 1] <= targets[:, 2]
)

print("\nTRAIN graph integrity: PASS")
print("TRAIN windows:", len(end_frames))
print("TRAIN episodes:", len(np.unique(episodes)))
print("Graph snapshots:", len(nodes_raw))

# =============================================================================
# 3. FIT TRAIN-ONLY PREPROCESSING
# =============================================================================

log_nodes = np.log1p(nodes_raw)

flat = log_nodes.reshape(-1, 8)

median = np.median(
    flat, axis=0
).astype(np.float32)

q25, q75 = np.percentile(
    flat,
    [25, 75],
    axis=0
)

scale = np.maximum(
    q75 - q25,
    0.05
).astype(np.float32)

nodes_scaled = np.clip(
    (log_nodes - median) / scale,
    -8.0,
    8.0
).astype(np.float32)

del nodes_raw, log_nodes, flat

# Same outgoing-row normalization used in GCN training.
outgoing = edges_raw.sum(
    axis=-1,
    keepdims=True
)

edges_scaled = (
    edges_raw /
    np.maximum(outgoing, 1e-8)
).astype(np.float32)

del edges_raw, outgoing

assert np.isfinite(nodes_scaled).all()
assert np.isfinite(edges_scaled).all()

print("\nTRAIN scaler: LOG1P + MEDIAN/IQR + CLIP8")
print("Edge normalization: OUTGOING ROW")
print("Node scaler fitted on TRAIN: YES")

# =============================================================================
# 4. DEFINE GAT + LSTM FORECASTER
# =============================================================================

class GATLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_features=8,
        gat_hidden=32,
        heads=2,
        lstm_hidden=64,
        dropout=0.1
    ):
        super().__init__()

        # Genuine graph attention layer.
        #
        # edge_dim=1 allows the attention mechanism
        # to consider each edge's normalized traffic.
        #
        # concat=False averages the attention heads,
        # keeping embedding dimension = gat_hidden.

        self.gat1 = GATConv(
            in_channels=input_features,
            out_channels=gat_hidden,
            heads=heads,
            concat=False,
            dropout=0.0,
            edge_dim=1,
            add_self_loops=True,
            fill_value=0.0
        )

        self.gat2 = GATConv(
            in_channels=gat_hidden,
            out_channels=gat_hidden,
            heads=heads,
            concat=False,
            dropout=0.0,
            edge_dim=1,
            add_self_loops=True,
            fill_value=0.0
        )

        self.dropout = nn.Dropout(
            dropout
        )

        # Mean + max graph pooling -> temporal LSTM.
        self.lstm = nn.LSTM(
            input_size=2 * gat_hidden,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    def forward(self, node_input, edge_input):

        # node_input: [B, 10, 8, 8]
        # edge_input: [B, 10, 8, 8]
        #
        # Edge orientation: [source, destination]

        b, t, n, f = node_input.shape

        graph_count = b * t

        # Flatten temporal graphs into a disconnected
        # PyTorch Geometric graph batch.

        x = node_input.reshape(
            graph_count * n,
            f
        )

        adjacency = edge_input.reshape(
            graph_count,
            n,
            n
        )

        # Extract directed nonzero edges.
        graph_id, src, dst = torch.where(
            adjacency > 0
        )

        offsets = graph_id * n

        edge_index = torch.stack(
            [
                offsets + src,
                offsets + dst
            ],
            dim=0
        ).long()

        # Edge traffic is an attention input.
        edge_attr = adjacency[
            graph_id,
            src,
            dst
        ].unsqueeze(-1)

        # First graph attention layer.
        h = self.gat1(
            x,
            edge_index,
            edge_attr=edge_attr
        )

        h = F.elu(h)
        h = self.dropout(h)

        # Second graph attention layer.
        h = self.gat2(
            h,
            edge_index,
            edge_attr=edge_attr
        )

        h = F.elu(h)

        # Recover separate graph snapshots.
        h = h.reshape(
            graph_count,
            n,
            -1
        )

        # Pool node embeddings at each minute.
        mean_pool = h.mean(dim=1)
        max_pool = h.max(dim=1).values

        graph_vectors = torch.cat(
            [
                mean_pool,
                max_pool
            ],
            dim=-1
        )

        graph_sequence = graph_vectors.reshape(
            b,
            t,
            -1
        )

        # Temporal forecasting.
        temporal_output, _ = self.lstm(
            graph_sequence
        )

        last_state = temporal_output[:, -1, :]

        logits = self.head(last_state)

        # Predict conditional hazards for intervals:
        # [0,5], (5,10], (10,15] minutes.
        hazards = torch.sigmoid(logits)

        # Convert to cumulative probabilities.
        # Monotonicity is guaranteed by construction.
        probabilities = (
            1.0 - torch.cumprod(
                1.0 - hazards,
                dim=1
            )
        )

        return probabilities.clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )


# =============================================================================
# 5. INITIALIZE MODEL AND OPTIMIZER
# =============================================================================

model = GATLSTMForecaster(
    input_features=8,
    gat_hidden=GAT_HIDDEN,
    heads=ATTENTION_HEADS,
    lstm_hidden=LSTM_HIDDEN,
    dropout=DROPOUT
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

criterion = nn.BCELoss()

parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nMODEL ARCHITECTURE")
print(model)

print(
    "\nTrainable parameters:",
    parameter_count
)

# =============================================================================
# 6. TEST ACTUAL GRAPH ATTENTION FORWARD PASS
# =============================================================================

model.eval()

with torch.inference_mode():

    dummy_nodes = torch.rand(
        2, 10, 8, 8,
        device=DEVICE
    )

    dummy_edges = torch.zeros(
        2, 10, 8, 8,
        device=DEVICE
    )

    # Directed ring edges.
    for src in range(8):
        dummy_edges[
            :, :, src, (src + 1) % 8
        ] = 1.0

    output = model(
        dummy_nodes,
        dummy_edges
    )

    assert output.shape == (2, 3)

    assert torch.isfinite(output).all()

    assert torch.all(
        output[:, 0] <= output[:, 1]
    )

    assert torch.all(
        output[:, 1] <= output[:, 2]
    )

    # Diagnostic: determine whether changing graph
    # connectivity affects the model's initial output.
    isolated = torch.zeros_like(dummy_edges)

    isolated_output = model(
        dummy_nodes,
        isolated
    )

    edge_sensitivity = (
        output - isolated_output
    ).abs().max().item()

print("\nGAT forward pass: PASS")
print("Horizon monotonicity: PASS")
print("Initial edge sensitivity:", edge_sensitivity)

# =============================================================================
# 7. CHECKPOINT RECOVERY
# =============================================================================

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

start_epoch = 0
loss_history = []

if FINAL_FILE.exists():

    saved = torch.load(
        FINAL_FILE,
        map_location="cpu",
        weights_only=False
    )

    if saved["graph_sha256"] != train_sha:
        raise RuntimeError(
            "Existing GAT checkpoint dataset mismatch."
        )

    if saved["config"] != CONFIG:
        raise RuntimeError(
            "Existing GAT checkpoint config mismatch."
        )

    print("\nGAT + LSTM already trained.")
    print("Checkpoint:", FINAL_FILE)
    print("SHA256:", sha256(FINAL_FILE))
    print("No retraining performed.")

else:

    if PROGRESS_FILE.exists():

        progress = torch.load(
            PROGRESS_FILE,
            map_location=DEVICE,
            weights_only=False
        )

        if progress["graph_sha256"] != train_sha:
            raise RuntimeError(
                "Progress checkpoint dataset mismatch."
            )

        if progress["config"] != CONFIG:
            raise RuntimeError(
                "Progress configuration mismatch."
            )

        if not np.array_equal(
            progress["median"],
            median
        ):
            raise RuntimeError(
                "Saved scaler median mismatch."
            )

        if not np.array_equal(
            progress["scale"],
            scale
        ):
            raise RuntimeError(
                "Saved scaler IQR mismatch."
            )

        model.load_state_dict(
            progress["model_state"]
        )

        optimizer.load_state_dict(
            progress["optimizer_state"]
        )

        start_epoch = int(
            progress["completed_epoch"]
        )

        loss_history = list(
            progress["loss_history"]
        )

        torch.set_rng_state(
            progress["torch_rng_state"].cpu()
        )

        if (
            DEVICE.type == "cuda"
            and "cuda_rng_state" in progress
        ):
            torch.cuda.set_rng_state_all(
                progress["cuda_rng_state"]
            )

        print(
            "\nResuming after epoch:",
            start_epoch
        )

    # =========================================================================
    # 8. TRAIN GAT + LSTM
    # =========================================================================

    n = len(end_frames)

    offsets = np.arange(
        -9,
        1,
        dtype=np.int64
    )

    print("\n" + "=" * 85)
    print("STARTING GAT + LSTM TRAINING")
    print("=" * 85)

    print("Windows:", n)
    print("Epochs:", EPOCHS)
    print("Batch size:", BATCH_SIZE)
    print("Device:", DEVICE)

    for epoch in range(
        start_epoch,
        EPOCHS
    ):

        started = time.monotonic()

        model.train()

        order = np.random.default_rng(
            SEED + epoch
        ).permutation(n)

        total_loss = 0.0
        total_seen = 0

        for start in range(
            0,
            n,
            BATCH_SIZE
        ):

            ids = order[
                start:start + BATCH_SIZE
            ]

            frame_ids = (
                end_frames[ids, None]
                + offsets[None, :]
            )

            xb_nodes = torch.from_numpy(
                nodes_scaled[frame_ids]
            ).to(DEVICE)

            xb_edges = torch.from_numpy(
                edges_scaled[frame_ids]
            ).to(DEVICE)

            yb = torch.from_numpy(
                targets[ids]
            ).to(DEVICE)

            optimizer.zero_grad(
                set_to_none=True
            )

            predictions = model(
                xb_nodes,
                xb_edges
            )

            loss = criterion(
                predictions,
                yb
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss at epoch {epoch + 1}"
                )

            loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_loss += (
                loss.item() * len(ids)
            )

            total_seen += len(ids)

        epoch_loss = total_loss / total_seen

        loss_history.append(
            float(epoch_loss)
        )

        duration = time.monotonic() - started

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS}"
            f" | TRAIN BCE={epoch_loss:.6f}"
            f" | seconds={duration:.1f}",
            flush=True
        )

        # Save progress after every completed epoch.
        progress = {
            "completed_epoch": epoch + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "median": median,
            "scale": scale,
            "graph_sha256": train_sha,
            "config": CONFIG,
            "loss_history": loss_history,
            "torch_rng_state": torch.get_rng_state()
        }

        if DEVICE.type == "cuda":
            progress["cuda_rng_state"] = (
                torch.cuda.get_rng_state_all()
            )

        atomic_save(
            progress,
            PROGRESS_FILE
        )

    # =========================================================================
    # 9. SAVE FINAL TRAINED CHECKPOINT
    # =========================================================================

    model.eval()

    final_checkpoint = {
        "model_state": model.state_dict(),
        "median": median,
        "scale": scale,
        "graph_sha256": train_sha,
        "config": CONFIG,
        "loss_history": loss_history,
        "architecture": "PyG_GATConv_GATConv_LSTM",
        "graph_origin": "SYNTHETIC",
        "data_split": "TRAIN_ONLY",
        "validation_accessed": False,
        "test_accessed": False,
        "torch_version": torch.__version__,
        "pyg_version": torch_geometric.__version__
    }

    atomic_save(
        final_checkpoint,
        FINAL_FILE
    )

    record = {
        "model": "GATLSTMForecaster",
        "architecture": "PyG_GATConv_GATConv_LSTM",
        "config": CONFIG,
        "parameters": parameter_count,
        "training_windows": n,
        "training_episodes": int(
            len(np.unique(episodes))
        ),
        "train_graph_sha256": train_sha,
        "checkpoint_sha256": sha256(FINAL_FILE),
        "training_bce": loss_history,
        "final_train_bce": loss_history[-1],
        "graph_origin": "SYNTHETIC",
        "official_validation_accessed": False,
        "official_test_accessed": False
    }

    temp_record = RECORD_FILE.with_name(
        RECORD_FILE.name + ".tmp"
    )

    temp_record.write_text(
        json.dumps(
            record,
            indent=2
        )
    )

    temp_record.replace(RECORD_FILE)

    print("\n" + "=" * 85)
    print("GAT + LSTM — TRAINING COMPLETE")
    print("=" * 85)

    print("Checkpoint:", FINAL_FILE)
    print("SHA256:", sha256(FINAL_FILE))
    print("Trainable parameters:", parameter_count)
    print("Training windows:", n)
    print("Final TRAIN BCE:", loss_history[-1])

    print("\nExisting checkpoints modified: NO")
    print("Official VALIDATION accessed: NO")
    print("Official TEST accessed: NO")

    print("=" * 85)

In [ ]:

# =============================================================================
# CELL 12 — GAT + LSTM OFFICIAL VALIDATION
#
# Compare:
#   1. Original Graph + LSTM
#   2. Optimized Graph + LSTM
#   3. GCN + LSTM
#   4. GAT + LSTM
#   5. Stacked LSTM (previous AP reference only)
#
# METRICS:
#   H5/H10/H15 AP, AUROC, Brier
#   Precision, Recall, F1, FPR @ 0.5
#   Episode-level false alerts
#   Paired episode-bootstrap AP differences
#
# No retraining, no new scaler, no TEST access.
# =============================================================================

from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GATConv

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

BASE = ROOT / "Synthetic_NAF_v2_GraphCompanion"

TRAIN_MANIFEST = (
    BASE / "TRAIN_only_v1"
    / "GRAPH_COMPANION_MANIFEST.json"
)

VAL_FILE = (
    BASE / "VALIDATION_only_v1"
    / "VALIDATION_temporal_graphs.npz"
)

VAL_MANIFEST = (
    BASE / "VALIDATION_only_v1"
    / "GRAPH_COMPANION_VALIDATION_MANIFEST.json"
)

FROZEN_VAL = (
    ROOT / "Synthetic_NAF_v2"
    / "release_candidate_corrected"
    / "windows_W10_VALIDATION.npz"
)

GAT_CHECKPOINT = (
    BASE / "models" / "GAT_LSTM"
    / "train_only_v1" / "GAT_LSTM_STATE.pt"
)

GCN_PRED_FILE = (
    BASE / "models" / "GCN_LSTM"
    / "validation_v1" / "GCN_VAL_PREDICTIONS.npz"
)

GCN_REPORT_FILE = (
    BASE / "models" / "GCN_LSTM"
    / "validation_v1" / "GCN_VAL_REPORT.json"
)

OUTPUT_DIR = (
    BASE / "models" / "GAT_LSTM"
    / "validation_v1"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 128
THRESHOLD = 0.50
BOOTSTRAP_REPEATS = 300
HORIZONS = [5, 10, 15]

if DEVICE.type == "cpu":
    torch.set_num_threads(
        min(4, os.cpu_count() or 1)
    )


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()


print("=" * 90)
print("CELL 12 — GAT + LSTM VALIDATION")
print("=" * 90)
print("Device:", DEVICE)

# =============================================================================
# 2. CHECK INPUT FILES AND DATASET INTEGRITY
# =============================================================================

required = [
    TRAIN_MANIFEST,
    VAL_FILE,
    VAL_MANIFEST,
    FROZEN_VAL,
    GAT_CHECKPOINT,
    GCN_PRED_FILE,
    GCN_REPORT_FILE
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)

train_meta = json.loads(
    TRAIN_MANIFEST.read_text()
)

val_meta = json.loads(
    VAL_MANIFEST.read_text()
)

gcn_report = json.loads(
    GCN_REPORT_FILE.read_text()
)

train_sha = train_meta["artifact_sha256"]
val_sha = sha256(VAL_FILE)
gat_sha = sha256(GAT_CHECKPOINT)

assert val_sha == val_meta["artifact_sha256"]

assert (
    val_meta["source_train_graph_sha256"]
    == train_sha
)

assert (
    val_meta["generator_seed"]
    == train_meta["seed"]
)

assert (
    gcn_report["validation_graph_sha256"]
    == val_sha
)

print("\nGraph provenance: PASS")

# =============================================================================
# 3. LOAD VALIDATION DATA
# =============================================================================

with np.load(VAL_FILE, allow_pickle=False) as d:

    nodes_raw = d["node_frames"].astype(np.float32)

    edges_raw = d["edge_flow_frames"].astype(np.float32)

    end_frames = d["window_end_frame"].astype(np.int64)

    y_true = d["y_horizons"].astype(np.int8)

    episodes = d["episode_id"].astype(str)

    minutes = d["minute"].astype(np.int32)

assert nodes_raw.shape[1:] == (8, 8)

assert edges_raw.shape == (
    len(nodes_raw), 8, 8
)

assert y_true.shape == (len(end_frames), 3)

assert len(episodes) == len(end_frames)
assert len(minutes) == len(end_frames)

assert end_frames.min() >= 9
assert end_frames.max() < len(nodes_raw)

assert np.isfinite(nodes_raw).all()
assert np.isfinite(edges_raw).all()

assert (nodes_raw >= 0).all()
assert (edges_raw >= 0).all()

assert np.isin(y_true, [0, 1]).all()

assert np.all(y_true[:, 0] <= y_true[:, 1])
assert np.all(y_true[:, 1] <= y_true[:, 2])

# Verify graph rows correspond exactly to frozen V2 validation.
with np.load(FROZEN_VAL, allow_pickle=False) as d:

    assert np.array_equal(
        y_true, d["y_horizons"]
    )

    assert np.array_equal(
        episodes, d["episode_id"].astype(str)
    )

    assert np.array_equal(
        minutes, d["minute"]
    )

print("Frozen validation label alignment: PASS")
print("Validation windows:", len(y_true))
print("Validation episodes:", len(np.unique(episodes)))
print("Graph snapshots:", len(nodes_raw))

# =============================================================================
# 4. LOAD PREVIOUS GCN AND GRAPH MODEL PREDICTIONS
# =============================================================================

with np.load(GCN_PRED_FILE, allow_pickle=False) as d:

    assert np.array_equal(
        y_true, d["y_horizons"]
    )

    assert np.array_equal(
        episodes, d["episode_id"].astype(str)
    )

    assert np.array_equal(
        minutes, d["minute"]
    )

    gcn_probs = d[
        "gcn_probabilities"
    ].astype(np.float64)

    original_probs = d[
        "original_graph_probabilities"
    ].astype(np.float64)

    optimized_probs = d[
        "optimized_graph_probabilities"
    ].astype(np.float64)

for p in [
    gcn_probs,
    original_probs,
    optimized_probs
]:
    assert p.shape == y_true.shape
    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()

print("Previous model prediction alignment: PASS")

# =============================================================================
# 5. RESTORE THE EXACT TRAINED GAT ARCHITECTURE
# =============================================================================

class GATLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_features=8,
        gat_hidden=32,
        heads=2,
        lstm_hidden=64,
        dropout=0.1
    ):
        super().__init__()

        self.gat1 = GATConv(
            in_channels=input_features,
            out_channels=gat_hidden,
            heads=heads,
            concat=False,
            dropout=0.0,
            edge_dim=1,
            add_self_loops=True,
            fill_value=0.0
        )

        self.gat2 = GATConv(
            in_channels=gat_hidden,
            out_channels=gat_hidden,
            heads=heads,
            concat=False,
            dropout=0.0,
            edge_dim=1,
            add_self_loops=True,
            fill_value=0.0
        )

        self.dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=2 * gat_hidden,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    def forward(self, node_input, edge_input):

        b, t, n, f = node_input.shape

        graph_count = b * t

        x = node_input.reshape(
            graph_count * n,
            f
        )

        adjacency = edge_input.reshape(
            graph_count,
            n,
            n
        )

        graph_id, src, dst = torch.where(
            adjacency > 0
        )

        offsets = graph_id * n

        edge_index = torch.stack(
            [
                offsets + src,
                offsets + dst
            ],
            dim=0
        ).long()

        edge_attr = adjacency[
            graph_id,
            src,
            dst
        ].unsqueeze(-1)

        h = self.gat1(
            x,
            edge_index,
            edge_attr=edge_attr
        )

        h = F.elu(h)
        h = self.dropout(h)

        h = self.gat2(
            h,
            edge_index,
            edge_attr=edge_attr
        )

        h = F.elu(h)

        h = h.reshape(
            graph_count,
            n,
            -1
        )

        graph_vectors = torch.cat(
            [
                h.mean(dim=1),
                h.max(dim=1).values
            ],
            dim=-1
        )

        sequence = graph_vectors.reshape(
            b,
            t,
            -1
        )

        temporal_output, _ = self.lstm(
            sequence
        )

        logits = self.head(
            temporal_output[:, -1, :]
        )

        hazards = torch.sigmoid(logits)

        return (
            1.0 - torch.cumprod(
                1.0 - hazards,
                dim=1
            )
        ).clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )

# =============================================================================
# 6. LOAD GAT CHECKPOINT AND TRAIN-FITTED SCALER
# =============================================================================

# Load only the checkpoint created by your own training code.
checkpoint = torch.load(
    GAT_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

assert checkpoint["graph_sha256"] == train_sha

cfg = checkpoint["config"]

assert cfg["architecture"] == "PyG_GATConv_GATConv_LSTM"
assert cfg["edge_processing"] == "OUTGOING_ROW_NORMALIZED"
assert cfg["gat_concat"] is False
assert cfg["gat_self_loop_fill"] == 0.0
assert cfg["nodes"] == 8
assert cfg["node_features"] == 8
assert cfg["history"] == 10

model = GATLSTMForecaster(
    input_features=cfg["node_features"],
    gat_hidden=cfg["gat_hidden"],
    heads=cfg["attention_heads"],
    lstm_hidden=cfg["lstm_hidden"],
    dropout=cfg["dropout"]
).to(DEVICE)

model.load_state_dict(
    checkpoint["model_state"],
    strict=True
)

model.eval()

median = np.asarray(
    checkpoint["median"],
    dtype=np.float32
)

scale = np.asarray(
    checkpoint["scale"],
    dtype=np.float32
)

assert median.shape == (8,)
assert scale.shape == (8,)

assert np.isfinite(median).all()
assert np.isfinite(scale).all()
assert (scale > 0).all()

print("\nGAT checkpoint: PASS")
print("GAT SHA256:", gat_sha)
print("Attention heads:", cfg["attention_heads"])

# =============================================================================
# 7. PREPROCESS EXACTLY AS IN TRAINING
# =============================================================================

nodes_scaled = np.clip(
    (np.log1p(nodes_raw) - median) / scale,
    -8.0,
    8.0
).astype(np.float32)

# Important: outgoing source-row normalization,
# the same procedure used during GAT training.
outgoing = edges_raw.sum(
    axis=-1,
    keepdims=True
)

edges_scaled = (
    edges_raw / np.maximum(outgoing, 1e-8)
).astype(np.float32)

assert np.isfinite(nodes_scaled).all()
assert np.isfinite(edges_scaled).all()

print("TRAIN-fitted preprocessing reused: PASS")
print("Scaler refitted on validation: NO")

# =============================================================================
# 8. GAT VALIDATION INFERENCE
# =============================================================================

frame_offsets = np.arange(
    -9, 1,
    dtype=np.int64
)

predictions = []

print("\nRunning GAT validation inference...")

with torch.inference_mode():

    for start in range(
        0,
        len(end_frames),
        BATCH_SIZE
    ):

        stop = min(
            start + BATCH_SIZE,
            len(end_frames)
        )

        ids = np.arange(start, stop)

        indices = (
            end_frames[ids, None]
            + frame_offsets[None, :]
        )

        xb = torch.from_numpy(
            nodes_scaled[indices]
        ).to(DEVICE)

        eb = torch.from_numpy(
            edges_scaled[indices]
        ).to(DEVICE)

        probs = model(xb, eb)

        predictions.append(
            probs.cpu().numpy()
        )

        if (
            stop % 5000 < BATCH_SIZE
            or stop == len(end_frames)
        ):
            print(
                f"  Predicted {stop:,}/{len(end_frames):,}",
                flush=True
            )

gat_probs = np.concatenate(
    predictions,
    axis=0
).astype(np.float64)

assert gat_probs.shape == y_true.shape
assert np.isfinite(gat_probs).all()
assert ((gat_probs >= 0) & (gat_probs <= 1)).all()

assert np.all(
    gat_probs[:, 0] <= gat_probs[:, 1] + 1e-7
)

assert np.all(
    gat_probs[:, 1] <= gat_probs[:, 2] + 1e-7
)

print("\nGAT inference: PASS")
print("Horizon monotonicity: PASS")

# =============================================================================
# 9. EVALUATE ALL GRAPH FORECASTERS
# =============================================================================

model_predictions = {
    "Original Graph LSTM": original_probs,
    "Optimized Graph LSTM": optimized_probs,
    "GCN + LSTM": gcn_probs,
    "GAT + LSTM": gat_probs
}

metric_rows = []

for name, probabilities in model_predictions.items():

    for j, horizon in enumerate(HORIZONS):

        y = y_true[:, j].astype(int)
        p = probabilities[:, j]

        predicted = (
            p >= THRESHOLD
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y,
            predicted,
            labels=[0, 1]
        ).ravel()

        metric_rows.append({
            "Model": name,
            "Horizon": f"H{horizon}",
            "AP": float(
                average_precision_score(y, p)
            ),
            "AUROC": float(
                roc_auc_score(y, p)
            ),
            "Brier": float(
                brier_score_loss(y, p)
            ),
            "Precision@0.5": float(
                precision_score(
                    y, predicted, zero_division=0
                )
            ),
            "Recall@0.5": float(
                recall_score(
                    y, predicted, zero_division=0
                )
            ),
            "F1@0.5": float(
                f1_score(
                    y, predicted, zero_division=0
                )
            ),
            "FPR@0.5": float(
                fp / max(fp + tn, 1)
            ),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn)
        })

metrics_df = pd.DataFrame(metric_rows)

print("\n" + "=" * 90)
print("GAT vs GCN vs CUSTOM GRAPH — VALIDATION RESULTS")
print("=" * 90)

print(
    metrics_df[
        [
            "Model",
            "Horizon",
            "AP",
            "AUROC",
            "Brier",
            "Precision@0.5",
            "Recall@0.5",
            "F1@0.5",
            "FPR@0.5"
        ]
    ].round(6).to_string(index=False)
)

# =============================================================================
# 10. AP COMPARISON WITH STACKED LSTM
# =============================================================================

stacked_ap = {
    "H5": 0.761005,
    "H10": 0.906018,
    "H15": 0.935854
}

comparison_rows = []

for j, h in enumerate(HORIZONS):

    row = {
        "Horizon": f"H{h}",
        "Stacked_LSTM_AP": stacked_ap[f"H{h}"]
    }

    for name, probs in model_predictions.items():
        row[name] = float(
            average_precision_score(
                y_true[:, j],
                probs[:, j]
            )
        )

    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

print("\n" + "=" * 90)
print("ALL GRAPH MODELS vs STACKED LSTM — AP")
print("=" * 90)

print(
    comparison_df.round(6).to_string(index=False)
)

print(
    "\nStacked LSTM values are earlier reported "
    "V2 validation AP, not replayed here."
)

# =============================================================================
# 11. EPISODE-LEVEL FALSE ALERTS
# =============================================================================

false_alert_rows = []

for j, h in enumerate(HORIZONS):

    frame = pd.DataFrame({
        "episode": episodes,
        "label": y_true[:, j],
        **{
            name: p[:, j]
            for name, p in model_predictions.items()
        }
    })

    aggregate = frame.groupby("episode").max()

    # Negative episode means no H-positive window
    # anywhere in the episode.
    negative_episodes = aggregate[
        aggregate["label"] == 0
    ]

    for name in model_predictions:

        false_alert_rows.append({
            "Model": name,
            "Horizon": f"H{h}",
            "Negative_episodes": int(
                len(negative_episodes)
            ),
            "False_alert_rate": (
                float(
                    (
                        negative_episodes[name]
                        >= THRESHOLD
                    ).mean()
                )
                if len(negative_episodes)
                else None
            )
        })

false_alert_df = pd.DataFrame(false_alert_rows)

print("\n" + "=" * 90)
print("EPISODE-LEVEL FALSE ALERTS AT THRESHOLD 0.5")
print("=" * 90)

print(
    false_alert_df.round(6).to_string(index=False)
)

# =============================================================================
# 12. PAIRED EPISODE BOOTSTRAP
#
# GAT vs GCN and GAT vs Original Graph LSTM
# Resample entire episodes rather than individual windows.
# =============================================================================

unique_episodes, episode_index = np.unique(
    episodes,
    return_inverse=True
)

rows_by_episode = [
    np.flatnonzero(episode_index == i)
    for i in range(len(unique_episodes))
]

rng = np.random.default_rng(20261027)

bootstrap_rows = []

print("\n" + "=" * 90)
print("PAIRED EPISODE BOOTSTRAP — AP DIFFERENCES")
print("=" * 90)

for reference_name in [
    "GCN + LSTM",
    "Original Graph LSTM"
]:

    reference_probs = model_predictions[
        reference_name
    ]

    for j, h in enumerate(HORIZONS):

        actual_delta = (
            average_precision_score(
                y_true[:, j],
                gat_probs[:, j]
            )
            -
            average_precision_score(
                y_true[:, j],
                reference_probs[:, j]
            )
        )

        delta_samples = []

        for _ in range(BOOTSTRAP_REPEATS):

            chosen = rng.integers(
                0,
                len(unique_episodes),
                size=len(unique_episodes)
            )

            sampled_rows = np.concatenate(
                [
                    rows_by_episode[k]
                    for k in chosen
                ]
            )

            ys = y_true[sampled_rows, j]

            if np.unique(ys).size != 2:
                continue

            ap_gat = average_precision_score(
                ys,
                gat_probs[sampled_rows, j]
            )

            ap_reference = average_precision_score(
                ys,
                reference_probs[sampled_rows, j]
            )

            delta_samples.append(
                ap_gat - ap_reference
            )

        if not delta_samples:
            raise RuntimeError(
                f"No bootstrap samples for H{h}"
            )

        low, high = np.percentile(
            delta_samples,
            [2.5, 97.5]
        )

        bootstrap_rows.append({
            "Comparison": f"GAT minus {reference_name}",
            "Horizon": f"H{h}",
            "Delta_AP": float(actual_delta),
            "CI95_lower": float(low),
            "CI95_upper": float(high),
            "Bootstrap_repeats": len(delta_samples)
        })

bootstrap_df = pd.DataFrame(bootstrap_rows)

print(
    bootstrap_df.round(6).to_string(index=False)
)

print(
    "\nPositive Delta_AP favors GAT. "
    "Intervals crossing zero indicate uncertain differences."
)

# =============================================================================
# 13. SAVE GAT VALIDATION ARTIFACTS
# =============================================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PRED_FILE = OUTPUT_DIR / "GAT_VAL_PREDICTIONS.npz"
METRICS_FILE = OUTPUT_DIR / "GAT_VAL_METRICS.csv"
REPORT_FILE = OUTPUT_DIR / "GAT_VAL_REPORT.json"

report = {
    "model": "PyG_GATConv_GATConv_LSTM",
    "gat_checkpoint_sha256": gat_sha,
    "validation_graph_sha256": val_sha,
    "gcn_checkpoint_sha256": (
        gcn_report["gcn_checkpoint_sha256"]
    ),
    "windows": int(len(end_frames)),
    "episodes": int(len(unique_episodes)),
    "threshold": THRESHOLD,
    "metrics": metric_rows,
    "ap_comparison": comparison_rows,
    "episode_false_alerts": false_alert_rows,
    "bootstrap": bootstrap_rows,
    "stacked_lstm_reference": stacked_ap,
    "graph_origin": "SYNTHETIC_FROM_AGGREGATE_V2",
    "validation_previously_examined": True,
    "test_accessed": False
}

output_paths = [
    PRED_FILE,
    METRICS_FILE,
    REPORT_FILE
]

if any(p.exists() for p in output_paths):

    if not all(p.exists() for p in output_paths):
        raise RuntimeError(
            "Partial GAT evaluation output exists. "
            "Inspect before rerunning."
        )

    previous = json.loads(
        REPORT_FILE.read_text()
    )

    for field in [
        "gat_checkpoint_sha256",
        "validation_graph_sha256",
        "gcn_checkpoint_sha256"
    ]:
        if previous[field] != report[field]:
            raise RuntimeError(
                f"Existing GAT report input mismatch: {field}"
            )

    print(
        "\nMatching GAT evaluation artifacts already exist. "
        "No overwrite performed."
    )

else:

    temp_pred = (
        OUTPUT_DIR / "GAT_VAL_PREDICTIONS.tmp.npz"
    )

    np.savez_compressed(
        temp_pred,
        y_horizons=y_true,
        episode_id=episodes,
        minute=minutes,
        gat_probabilities=gat_probs,
        gcn_probabilities=gcn_probs,
        original_graph_probabilities=original_probs,
        optimized_graph_probabilities=optimized_probs
    )

    temp_pred.replace(PRED_FILE)

    metrics_df.to_csv(
        METRICS_FILE,
        index=False
    )

    temporary_report = REPORT_FILE.with_name(
        REPORT_FILE.name + ".tmp"
    )

    temporary_report.write_text(
        json.dumps(report, indent=2)
    )

    temporary_report.replace(REPORT_FILE)

    print("\nGAT evaluation artifacts saved.")

print("\nPredictions:", PRED_FILE)
print("Metrics:", METRICS_FILE)
print("Report:", REPORT_FILE)

# =============================================================================
# 14. FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("GAT + LSTM VALIDATION COMPLETE")
print("=" * 90)

print("GAT evaluated: YES")
print("GCN compared: YES")
print("Original Graph + LSTM compared: YES")
print("Optimized Graph + LSTM compared: YES")
print("Stacked LSTM reference included: YES")
print("Official VALIDATION windows:", len(end_frames))
print("TRAIN checkpoints modified: NO")
print("Scaler refitted: NO")
print("Official TEST accessed: NO")

print("=" * 90)

